# SCI1402 — Analyse des homicides et extraction NLP

**Projet de science des données — TÉLUQ**

Ce carnet rassemble l'exploration des données, les analyses statistiques, le regroupement exploratoire, l'annotation humaine et l'évaluation du NLP.

**Lecture importante :** les catégories NLP représentent des mentions explicites dans les descriptions (`Notes`), et non une preuve de présence ou d'absence d'un comportement dans la réalité. Le jeu de test doit rester indépendant des ajustements des règles.

## Préparation

- Exécuter dans Kaggle avec les jeux de données requis attachés au notebook.
- Certaines cellules utilisent des chemins absolus `/kaggle/working` et nécessitent des fichiers produits lors des étapes précédentes.
- Les règles V2 gelées sont également versionnées séparément dans `src/`.
- Le carnet est conservé pour traçabilité : **son exécution intégrale depuis une session vierge n'a pas encore été validée**.

---


In [ ]:

from pathlib import Path
import pandas as pd

# Répertoire contenant les datasets Kaggle
data_dir = Path("/kaggle/input")

# Afficher tous les fichiers CSV disponibles
csv_files = sorted(data_dir.rglob("*.csv"))

print("Fichiers CSV disponibles :")
for file in csv_files:
    print(f" - {file}")

if not csv_files:
    raise FileNotFoundError(
        "Aucun fichier CSV trouvé. "
        "Ajoute le dataset au notebook."
    )

# Sélection du fichier principal
matches = [
    file for file in csv_files
    if file.name.lower() == "database.csv"
]

if not matches:
    raise FileNotFoundError(
        "database.csv introuvable. "
        "Vérifie les fichiers affichés ci-dessus."
    )

# Chargement
df = pd.read_csv(matches[0], low_memory=False)

# Inspection initiale
print("\nDimensions :", df.shape)
print("\nColonnes :")
print(df.columns.tolist())

print("\nAperçu :")
display(df.head())

print("\nValeurs manquantes (%) :")
display(
    (df.isna().mean() * 100)
    .sort_values(ascending=False)
    .round(2)
)


### 2. Audit de qualité des données

**Objectif :** examiner la qualité des 638 454 enregistrements afin de déterminer quelles variables pourront être utilisées dans les analyses statistiques et les modèles d'apprentissage automatique.

Bien qu'aucune valeur manquante (`NaN`) n'ait été détectée lors du chargement, certaines variables contiennent des valeurs telles que `Unknown` ou des codes numériques particuliers.

Nous allons identifier ces valeurs, mesurer leur fréquence et vérifier la cohérence des âges et des catégories.

Cette étape permettra d'éviter d'introduire des biais liés à des données inconnues ou incorrectement interprétées.

In [ ]:

import pandas as pd
import numpy as np

# Copie de travail : préserver le dataset original
df_clean = df.copy()

# 1. Identifier les valeurs inconnues
unknown_values = [
    "Unknown",
    "Not Specified",
    "Not Reported",
    "Missing",
    ""
]

unknown_stats = []

for col in df_clean.columns:
    if df_clean[col].dtype == "object":
        values = df_clean[col].astype("string").str.strip()

        count = values.isin(unknown_values).sum()

        if count > 0:
            unknown_stats.append({
                "Variable": col,
                "Inconnues": int(count),
                "Pourcentage": round(
                    count / len(df_clean) * 100, 2
                )
            })

unknown_df = pd.DataFrame(unknown_stats)

if not unknown_df.empty:
    unknown_df = unknown_df.sort_values(
        "Pourcentage", ascending=False
    )

print("VALEURS INCONNUES")
display(unknown_df)

# 2. Vérification des âges
for col in ["Victim Age", "Perpetrator Age"]:
    ages = pd.to_numeric(df_clean[col], errors="coerce")

    print(f"\n{col}")
    print("Valeurs non numériques :", ages.isna().sum())
    print("Âges égaux à zéro :", (ages == 0).sum())
    print("Âges supérieurs à 100 :", (ages > 100).sum())

    print("\nValeurs les plus fréquentes :")
    display(df_clean[col].value_counts().head(10))

# 3. Distribution des affaires résolues
print("\nSTATUT DES AFFAIRES")
display(
    df_clean["Crime Solved"].value_counts(
        dropna=False,
        normalize=True
    ).mul(100).round(2)
)

# 4. Types de crimes
print("\nTYPES DE CRIMES")
display(
    df_clean["Crime Type"].value_counts(
        dropna=False
    )
)

# 5. Vérification des doublons
print("\nDOUBLONS")
print(
    "Record ID dupliqués :",
    df_clean["Record ID"].duplicated().sum()
)

print(
    "Lignes entièrement identiques :",
    df_clean.duplicated().sum()
)


### 3. Préparation et segmentation des données

Cette étape consiste à préparer deux ensembles de données adaptés aux objectifs du projet.

Le premier regroupera les homicides intentionnels, indépendamment de leur statut de résolution. Le second contiendra les affaires résolues pour lesquelles certaines caractéristiques de l'auteur sont documentées.

Les codes correspondant à des informations inconnues seront traités comme des valeurs manquantes. Les âges feront l'objet d'une vérification particulière afin de distinguer les valeurs plausibles des codes spéciaux.

Les variables originales seront conservées dans un jeu de données brut pour assurer la traçabilité des transformations.

Cette préparation vise à obtenir des données cohérentes, sans introduire de valeurs artificielles susceptibles de fausser les analyses statistiques.

In [ ]:

import pandas as pd
import numpy as np

# ============================================
# SCI1402 - Préparation des données
# ============================================

# 1. Préserver les données originales
df_clean = df.copy()

# 2. Vérifier les codes d'âge particuliers
print("CODES D'ÂGE À VÉRIFIER")

for col in ["Victim Age", "Perpetrator Age"]:
    numeric = pd.to_numeric(
        df_clean[col], errors="coerce"
    )

    print(f"\n{col}")
    print("Valeurs >= 100 :")
    print(
        numeric[numeric >= 100]
        .value_counts()
        .sort_index()
    )

# 3. Normaliser les valeurs textuelles inconnues
unknown_values = [
    "Unknown",
    "Not Specified",
    "Not Reported",
    "Missing",
    ""
]

for col in df_clean.select_dtypes(
    include=["object", "string"]
).columns:

    df_clean[col] = (
        df_clean[col]
        .astype("string")
        .str.strip()
    )

    df_clean[col] = df_clean[col].replace(
        unknown_values, pd.NA
    )

# 4. Préparer les âges
# Les valeurs 0 sont traitées comme inconnues.
# Les valeurs >= 100 sont conservées à part
# pour vérification des conventions du dataset.

for col in ["Victim Age", "Perpetrator Age"]:

    numeric = pd.to_numeric(
        df_clean[col], errors="coerce"
    )

    df_clean[col + " Raw"] = numeric

    df_clean[col] = numeric.mask(
        (numeric <= 0) | (numeric >= 100)
    )

# 5. Sélectionner les homicides intentionnels
df_homicides = df_clean[
    df_clean["Crime Type"] == "Murder or Manslaughter"
].copy()

# 6. Sélectionner les affaires résolues
df_known = df_homicides[
    df_homicides["Crime Solved"] == "Yes"
].copy()

# 7. Résumé des ensembles
summary = pd.DataFrame({
    "Ensemble": [
        "Données originales",
        "Homicides intentionnels",
        "Affaires résolues"
    ],
    "Enregistrements": [
        len(df),
        len(df_homicides),
        len(df_known)
    ]
})

summary["Pourcentage du total"] = (
    summary["Enregistrements"]
    / len(df) * 100
).round(2)

print("\nRÉSUMÉ DES DATASETS")
display(summary)

# 8. Vérifier la disponibilité des variables
variables = [
    "Victim Age",
    "Victim Sex",
    "Perpetrator Age",
    "Perpetrator Sex",
    "Relationship",
    "Weapon"
]

availability = pd.DataFrame({
    "Variable": variables,
    "Disponibilité (%)": [
        round(
            df_known[col].notna().mean() * 100,
            2
        )
        for col in variables
    ]
})

print("\nDISPONIBILITÉ DES VARIABLES")
display(availability)

# 9. Sauvegarder les résultats
output_dir = "/kaggle/working/"

df_homicides.to_csv(
    output_dir + "homicides_clean.csv",
    index=False
)

df_known.to_csv(
    output_dir + "homicides_solved.csv",
    index=False
)

print("\nFichiers CSV créés avec succès.")


### 4. Analyse exploratoire des caractéristiques criminelles

L'analyse exploratoire des données (EDA) vise à comprendre les distributions et les relations entre les variables avant de développer des modèles d'apprentissage automatique.

Cette étape étudie les caractéristiques des homicides intentionnels documentés entre 1980 et 2014, notamment l'âge des auteurs, les armes utilisées et la relation entre la victime et l'auteur.

Les analyses portant sur les caractéristiques des auteurs sont limitées aux affaires résolues, afin de réduire la proportion d'informations inconnues.

Les résultats permettront d'identifier des associations statistiques et de déterminer quelles variables pourraient contribuer à une future analyse de similarité entre crimes.

Ces associations ne doivent pas être interprétées comme des relations causales ni comme des indicateurs individuels de dangerosité.

In [ ]:

# ============================================
# SCI1402 - Étape 4
# Analyse exploratoire des homicides
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration graphique
sns.set_theme(style="whitegrid")

# Copie des affaires résolues
eda = df_known.copy()

# ============================================
# 1. Distribution des âges des auteurs
# ============================================

ages = pd.to_numeric(
    eda["Perpetrator Age"],
    errors="coerce"
).dropna()

print("STATISTIQUES SUR L'ÂGE DES AUTEURS")
print(ages.describe())

plt.figure(figsize=(12, 5))

sns.histplot(
    ages,
    bins=40,
    kde=True
)

plt.title("Distribution des âges des auteurs")
plt.xlabel("Âge")
plt.ylabel("Nombre d'enregistrements")
plt.tight_layout()
plt.show()

# ============================================
# 2. Armes les plus fréquentes
# ============================================

weapons = (
    eda["Weapon"]
    .value_counts()
    .head(12)
)

print("\nARMES LES PLUS FRÉQUENTES")
display(weapons.to_frame("Enregistrements"))

plt.figure(figsize=(12, 6))

sns.barplot(
    x=weapons.values,
    y=weapons.index,
    color="steelblue"
)

plt.title("Armes utilisées dans les homicides résolus")
plt.xlabel("Nombre d'enregistrements")
plt.ylabel("Arme")
plt.tight_layout()
plt.show()

# ============================================
# 3. Relations victime-auteur
# ============================================

relationships = (
    eda["Relationship"]
    .value_counts()
    .head(12)
)

print("\nRELATIONS VICTIME-AUTEUR")
display(
    relationships.to_frame("Enregistrements")
)

plt.figure(figsize=(12, 6))

sns.barplot(
    x=relationships.values,
    y=relationships.index,
    color="slateblue"
)

plt.title("Relations entre victimes et auteurs")
plt.xlabel("Nombre d'enregistrements")
plt.ylabel("Relation")
plt.tight_layout()
plt.show()

# ============================================
# 4. Relation victime-auteur et arme
# ============================================

top_relations = (
    eda["Relationship"]
    .value_counts()
    .head(8)
    .index
)

top_weapons = (
    eda["Weapon"]
    .value_counts()
    .head(8)
    .index
)

subset = eda[
    eda["Relationship"].isin(top_relations)
    & eda["Weapon"].isin(top_weapons)
].copy()

cross_tab = pd.crosstab(
    subset["Relationship"],
    subset["Weapon"],
    normalize="index"
) * 100

print("\nRELATION ET ARME (%)")
display(cross_tab.round(2))

plt.figure(figsize=(13, 7))

sns.heatmap(
    cross_tab,
    annot=True,
    fmt=".1f",
    cmap="Blues"
)

plt.title(
    "Répartition des armes selon la relation "
    "victime-auteur (%)"
)

plt.xlabel("Arme")
plt.ylabel("Relation")
plt.tight_layout()
plt.show()

# ============================================
# 5. Évolution annuelle des homicides
# ============================================

annual = (
    df_homicides
    .groupby("Year")
    .size()
    .sort_index()
)

print("\nHOMICIDES PAR ANNÉE")
display(annual.to_frame("Enregistrements"))

plt.figure(figsize=(12, 5))

plt.plot(
    annual.index,
    annual.values,
    marker="o",
    markersize=3
)

plt.title("Évolution annuelle des homicides")
plt.xlabel("Année")
plt.ylabel("Nombre d'enregistrements")
plt.tight_layout()
plt.show()

# ============================================
# 6. Exportation des tableaux
# ============================================

weapons.to_csv(
    "/kaggle/working/weapon_distribution.csv"
)

relationships.to_csv(
    "/kaggle/working/relationship_distribution.csv"
)

cross_tab.to_csv(
    "/kaggle/working/relationship_weapon_matrix.csv"
)

annual.to_csv(
    "/kaggle/working/annual_homicides.csv"
)

print("\nAnalyse exploratoire terminée.")
print("Tableaux exportés dans /kaggle/working/")


### 5. Analyse statistique des associations entre variables criminelles

Cette étape vise à quantifier les associations entre certaines caractéristiques documentées des homicides.

Nous utiliserons le test du khi carré d'indépendance et le coefficient V de Cramér pour comparer plusieurs variables catégorielles.

Les analyses porteront sur les affaires résolues et examineront notamment les associations entre la relation victime-auteur, l'arme utilisée et les catégories d'âge.

Les valeurs manquantes seront exclues uniquement pour les variables impliquées dans chaque comparaison.

Les résultats permettront d'évaluer la force des associations, sans présumer qu'elles sont causales ou qu'elles permettent d'identifier un individu dangereux.

In [ ]:

# ============================================
# SCI1402 - Étape 5
# Associations statistiques
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency

# Copie des affaires résolues
analysis = df_known.copy()

# --------------------------------------------
# 1. Regroupement des armes
# --------------------------------------------

weapon_groups = {
    "Handgun": "Firearm",
    "Rifle": "Firearm",
    "Shotgun": "Firearm",
    "Firearm": "Firearm",
    "Gun": "Firearm",
    "Knife": "Knife",
    "Blunt Object": "Blunt Object",
    "Strangulation": "Asphyxiation",
    "Suffocation": "Asphyxiation",
    "Drowning": "Asphyxiation",
    "Fire": "Fire",
    "Drugs": "Drugs"
}

analysis["Weapon Group"] = (
    analysis["Weapon"]
    .map(weapon_groups)
    .fillna("Other")
)

analysis.loc[
    analysis["Weapon"].isna(),
    "Weapon Group"
] = pd.NA

# --------------------------------------------
# 2. Catégories d'âge
# --------------------------------------------

analysis["Age Group"] = pd.cut(
    pd.to_numeric(
        analysis["Perpetrator Age"],
        errors="coerce"
    ),
    bins=[0, 17, 24, 34, 44, 59, 99],
    labels=[
        "1-17",
        "18-24",
        "25-34",
        "35-44",
        "45-59",
        "60-99"
    ]
)

# --------------------------------------------
# 3. Fonction V de Cramér
# --------------------------------------------

def cramers_v(data, col1, col2):

    subset = data[
        [col1, col2]
    ].dropna()

    table = pd.crosstab(
        subset[col1],
        subset[col2]
    )

    if min(table.shape) < 2:
        return {
            "Variable 1": col1,
            "Variable 2": col2,
            "N": len(subset),
            "V de Cramér": np.nan,
            "p-value": np.nan
        }

    chi2, p, dof, expected = chi2_contingency(
        table,
        correction=False
    )

    n = table.to_numpy().sum()
    r, k = table.shape

    # V de Cramér corrigé du biais
    phi2 = chi2 / n

    phi2_corr = max(
        0,
        phi2 - ((k - 1) * (r - 1)) / (n - 1)
    )

    r_corr = r - ((r - 1) ** 2) / (n - 1)
    k_corr = k - ((k - 1) ** 2) / (n - 1)

    denominator = min(
        k_corr - 1,
        r_corr - 1
    )

    v = (
        np.sqrt(phi2_corr / denominator)
        if denominator > 0
        else np.nan
    )

    return {
        "Variable 1": col1,
        "Variable 2": col2,
        "N": n,
        "V de Cramér": round(v, 4),
        "p-value": p,
        "Cellules attendues < 5 (%)": round(
            (expected < 5).mean() * 100, 2
        )
    }

# --------------------------------------------
# 4. Comparaisons
# --------------------------------------------

comparisons = [
    ("Relationship", "Weapon Group"),
    ("Age Group", "Weapon Group"),
    ("Victim Sex", "Weapon Group"),
    ("Age Group", "Relationship")
]

results = []

for col1, col2 in comparisons:
    results.append(
        cramers_v(analysis, col1, col2)
    )

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "V de Cramér",
    ascending=False
)

print("ASSOCIATIONS STATISTIQUES")
display(results_df)

# --------------------------------------------
# 5. Visualisation
# --------------------------------------------

labels = (
    results_df["Variable 1"]
    + " / "
    + results_df["Variable 2"]
)

plt.figure(figsize=(11, 5))

sns.barplot(
    x=results_df["V de Cramér"],
    y=labels,
    color="steelblue"
)

plt.title("Force des associations - V de Cramér")
plt.xlabel("V de Cramér corrigé")
plt.ylabel("Variables comparées")

plt.xlim(0, 1)
plt.tight_layout()
plt.show()

# --------------------------------------------
# 6. Exportation
# --------------------------------------------

results_df.to_csv(
    "/kaggle/working/cramers_v_results.csv",
    index=False
)

print("Analyse statistique terminée.")


In [ ]:

# ============================================
# SCI1402 - Étape 6
# Exploration du dataset Serial Killers
# ============================================

from pathlib import Path
import pandas as pd

# Localiser les fichiers du nouveau dataset
base_path = Path("/kaggle/input")

serial_files = [
    p for p in base_path.rglob("*.csv")
    if "serial" in str(p).lower()
    or "dante890b" in str(p).lower()
]

print("FICHIERS TROUVÉS :")

for file in serial_files:
    print(file)

if not serial_files:
    raise FileNotFoundError(
        "Aucun CSV du dataset Serial Killers trouvé."
    )

# Charger le fichier
serial_df = pd.read_csv(
    serial_files[0],
    low_memory=False
)

# Dimensions
print("\nDIMENSIONS :")
print(serial_df.shape)

# Colonnes
print("\nCOLONNES :")
print(serial_df.columns.tolist())

# Aperçu
print("\nAPERÇU DES DONNÉES :")
display(serial_df.head(10))

# Types de données
print("\nTYPES DE VARIABLES :")
display(serial_df.dtypes.to_frame("Type"))

# Valeurs manquantes
print("\nVALEURS MANQUANTES :")

missing = pd.DataFrame({
    "Manquantes": serial_df.isna().sum(),
    "Pourcentage": (
        serial_df.isna().mean() * 100
    ).round(2)
})

display(
    missing.sort_values(
        "Pourcentage",
        ascending=False
    )
)

# Doublons
print("\nDOUBLONS COMPLETS :")
print(serial_df.duplicated().sum())

print("\nEXPLORATION TERMINÉE")


### 6.2. Exploration du dataset des tueurs en série selon le nombre de victimes

Le dataset *Wikipedia Serial Killers List* comprend deux fichiers CSV distincts.

Le premier contient 711 profils biographiques, tandis que le second semble regrouper des informations sur les tueurs en série selon leurs victimes.

Cette étape vise à examiner la structure du second fichier, identifier les variables disponibles et évaluer la possibilité de relier les deux ensembles.

Une attention particulière sera accordée à la qualité des identifiants, aux doublons et à la disponibilité des variables pertinentes pour l'analyse statistique.

L'objectif est de déterminer si ces données permettent de construire une table analytique cohérente avant de procéder au clustering.

In [ ]:

# ============================================
# SCI1402 - Étape 7
# Exploration du second dataset
# ============================================

from pathlib import Path
import pandas as pd
import numpy as np
import unicodedata
import re

# Chemin du dataset
dataset_path = Path(
    "/kaggle/input/datasets/"
    "dante890b/wikipedia-serial-killers-list"
)

# Chargement explicite des deux fichiers
profiles = pd.read_csv(
    dataset_path / "profile_killers.csv",
    low_memory=False
)

victims = pd.read_csv(
    dataset_path / "serial_killers_by_victims.csv",
    low_memory=False
)

# --------------------------------------------
# 1. Structure du second fichier
# --------------------------------------------

print("DIMENSIONS")
print("Profils :", profiles.shape)
print("Victimes :", victims.shape)

print("\nCOLONNES DU SECOND FICHIER")
print(victims.columns.tolist())

print("\nAPERÇU")
display(victims.head(10))

print("\nTYPES DE VARIABLES")
display(victims.dtypes.to_frame("Type"))

# --------------------------------------------
# 2. Valeurs manquantes
# --------------------------------------------

missing = pd.DataFrame({
    "Manquantes": victims.isna().sum(),
    "Pourcentage": (
        victims.isna().mean() * 100
    ).round(2)
})

print("\nVALEURS MANQUANTES")
display(
    missing.sort_values(
        "Pourcentage",
        ascending=False
    )
)

# --------------------------------------------
# 3. Doublons
# --------------------------------------------

print("\nDOUBLONS COMPLETS")
print(victims.duplicated().sum())

# --------------------------------------------
# 4. Identifier les colonnes communes
# --------------------------------------------

common_columns = sorted(
    set(profiles.columns)
    .intersection(victims.columns)
)

print("\nCOLONNES COMMUNES")
print(common_columns)

# --------------------------------------------
# 5. Vérifier la présence des noms
# --------------------------------------------

def normalize_name(value):

    if pd.isna(value):
        return pd.NA

    value = str(value).strip().casefold()

    value = unicodedata.normalize(
        "NFKD", value
    )

    value = "".join(
        c for c in value
        if not unicodedata.combining(c)
    )

    value = re.sub(
        r"[^a-z0-9]+", " ", value
    )

    return re.sub(
        r"\s+", " ", value
    ).strip()

if "Name" in victims.columns:

    profiles["Name Key"] = (
        profiles["Name"].map(normalize_name)
    )

    victims["Name Key"] = (
        victims["Name"].map(normalize_name)
    )

    # Exclure les clés manquantes
    p_keys = profiles["Name Key"].dropna()
    v_keys = victims["Name Key"].dropna()

    print("\nNOMS DISTINCTS")
    print("Profils :", p_keys.nunique())
    print("Victimes :", v_keys.nunique())

    print("\nCLÉS DUPLIQUÉES")
    print("Profils :", p_keys.duplicated().sum())
    print("Victimes :", v_keys.duplicated().sum())

    # Comparer les noms présents
    common_names = (
        set(p_keys)
        .intersection(set(v_keys))
    )

    print("\nNOMS COMMUNS")
    print(len(common_names))

    print("\nPROPORTION DES PROFILS APPARIÉS")
    print(
        round(
            len(set(p_keys) & set(v_keys))
            / max(p_keys.nunique(), 1)
            * 100,
            2
        ),
        "%"
    )

    # Afficher quelques noms non appariés
    unmatched = profiles[
        ~profiles["Name Key"].isin(common_names)
    ]

    print("\nEXEMPLES NON APPARIÉS")
    display(unmatched[["Name"]].head(15))

else:
    print(
        "\nLa colonne Name est absente du "
        "second fichier. Il faudra identifier "
        "une autre clé de correspondance."
    )

print("\nEXPLORATION TERMINÉE")


### 7. Construction du dataset analytique des tueurs en série

Cette étape consiste à fusionner les données biographiques avec les données relatives aux victimes et aux périodes d'activité.

Les noms normalisés seront utilisés comme clés de correspondance. Avant la fusion, nous vérifierons l'unicité des identifiants afin d'éviter les associations multiples non désirées.

De nouvelles variables seront ensuite calculées : durée d'activité, âge estimé au début des crimes, âge à l'arrestation, nombre de pays d'activité et écart entre victimes confirmées et possibles.

Les données originales seront conservées. Les valeurs manquantes ou incohérentes ne seront pas remplacées par des valeurs arbitraires.

Le dataset résultant constituera la base des futures analyses exploratoires et des regroupements statistiques de profils historiques.

In [ ]:

# ============================================
# SCI1402 - Étape 8
# Fusion et enrichissement des profils
# ============================================

import pandas as pd
import numpy as np
from pathlib import Path
import re
import unicodedata

OUTPUT = Path("/kaggle/working")
OUTPUT.mkdir(parents=True, exist_ok=True)

# --------------------------------------------
# 1. Normalisation des noms
# --------------------------------------------

def normalize_name(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().casefold()
    value = unicodedata.normalize("NFKD", value)

    value = "".join(
        c for c in value
        if not unicodedata.combining(c)
    )

    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

p = profiles.drop(
    columns=["Name Key"], errors="ignore"
).copy()

v = victims.drop(
    columns=["Name Key"], errors="ignore"
).copy()

p["Name Key"] = p["Name"].map(normalize_name)
v["Name Key"] = v["Name"].map(normalize_name)

# --------------------------------------------
# 2. Validation des clés
# --------------------------------------------

for label, data in [("Profils", p), ("Victimes", v)]:
    if data["Name Key"].isna().any():
        raise ValueError(
            f"{label} : noms manquants."
        )

    if data["Name Key"].duplicated().any():
        duplicates = data.loc[
            data["Name Key"].duplicated(False),
            "Name"
        ]
        print(duplicates.to_list())
        raise ValueError(
            f"{label} : clés non uniques."
        )

# --------------------------------------------
# 3. Audit des correspondances
# --------------------------------------------

audit = p[["Name", "Name Key"]].merge(
    v[["Name", "Name Key"]],
    on="Name Key",
    how="outer",
    suffixes=("_profile", "_victims"),
    indicator=True,
    validate="one_to_one"
)

print("RÉSULTAT DES CORRESPONDANCES")
display(audit["_merge"].value_counts())

display(
    audit.loc[
        audit["_merge"] != "both",
        ["Name_profile", "Name_victims", "_merge"]
    ].head(30)
)

audit.to_csv(
    OUTPUT / "serial_killers_match_audit.csv",
    index=False
)

# --------------------------------------------
# 4. Fusion des profils
# --------------------------------------------

serial = p.merge(
    v.drop(columns=["Name"]),
    on="Name Key",
    how="inner",
    validate="one_to_one"
)

print("\nDIMENSIONS APRÈS FUSION")
print(serial.shape)

# --------------------------------------------
# 5. Conversion des dates
# --------------------------------------------

date_cols = [
    "Born Date",
    "Died Date",
    "Date Apprehended"
]

for col in date_cols:
    serial[col] = pd.to_datetime(
        serial[col],
        errors="coerce",
        format="mixed"
    )

# --------------------------------------------
# 6. Conversion des variables numériques
# --------------------------------------------

numeric_cols = [
    "Start year",
    "End year",
    "Proven victims",
    "Possible victims"
]

for col in numeric_cols:
    serial[col] = pd.to_numeric(
        serial[col],
        errors="coerce"
    )

# --------------------------------------------
# 7. Durée d'activité inclusive
# --------------------------------------------

serial["Activity Years"] = (
    serial["End year"]
    - serial["Start year"]
    + 1
)

serial.loc[
    serial["Activity Years"] <= 0,
    "Activity Years"
] = np.nan

# --------------------------------------------
# 8. Âge estimé au début des crimes
# --------------------------------------------

serial["Age at Start"] = (
    serial["Start year"]
    - serial["Born Date"].dt.year
)

# Comme seul l'année du premier crime est
# disponible, l'âge peut varier d'un an.

serial.loc[
    ~serial["Age at Start"].between(10, 100),
    "Age at Start"
] = np.nan

# --------------------------------------------
# 9. Âge à l'arrestation
# --------------------------------------------

born = serial["Born Date"]
arrested = serial["Date Apprehended"]

serial["Age at Arrest"] = (
    arrested.dt.year - born.dt.year
    - (
        (arrested.dt.month < born.dt.month)
        | (
            (arrested.dt.month == born.dt.month)
            & (arrested.dt.day < born.dt.day)
        )
    ).astype(int)
)

serial.loc[
    ~serial["Age at Arrest"].between(10, 110),
    "Age at Arrest"
] = np.nan

# --------------------------------------------
# 10. Victimes par année calendaire active
# --------------------------------------------

serial["Victims per Active Year"] = (
    serial["Proven victims"]
    / serial["Activity Years"]
)

# Il s'agit d'une moyenne sur les années
# calendaires inclusives, et non d'une
# fréquence réelle de commission des crimes.

# --------------------------------------------
# 11. Écart entre victimes possibles
#     et confirmées
# --------------------------------------------

serial["Victim Gap"] = (
    serial["Possible victims"]
    - serial["Proven victims"]
)

# --------------------------------------------
# 12. Nombre de pays déclarés
# --------------------------------------------

def count_countries(value):
    if pd.isna(value):
        return np.nan

    countries = [
        x.strip()
        for x in str(value).split(",")
        if x.strip()
    ]

    return len(set(countries))

serial["Country Count"] = (
    serial["Country"].apply(count_countries)
)

# --------------------------------------------
# 13. Contrôles de cohérence
# --------------------------------------------

print("\nANOMALIES POTENTIELLES")

print(
    "Victimes possibles < confirmées :",
    (serial["Victim Gap"] < 0).sum()
)

print(
    "Périodes inversées :",
    (
        serial["End year"]
        < serial["Start year"]
    ).sum()
)

print(
    "Débuts d'activité manquants :",
    serial["Start year"].isna().sum()
)

# --------------------------------------------
# 14. Disponibilité des variables dérivées
# --------------------------------------------

features = [
    "Proven victims",
    "Possible victims",
    "Activity Years",
    "Age at Start",
    "Age at Arrest",
    "Victims per Active Year",
    "Victim Gap",
    "Country Count"
]

availability = pd.DataFrame({
    "Variable": features,
    "Disponibles": [
        serial[col].notna().sum()
        for col in features
    ],
    "Disponibilité (%)": [
        round(
            serial[col].notna().mean() * 100,
            2
        )
        for col in features
    ]
})

print("\nDISPONIBILITÉ DES VARIABLES")
display(availability)

# --------------------------------------------
# 15. Aperçu analytique
# --------------------------------------------

preview_cols = [
    "Name",
    "Country",
    "Proven victims",
    "Possible victims",
    "Activity Years",
    "Age at Start",
    "Age at Arrest",
    "Victims per Active Year",
    "Victim Gap",
    "Country Count"
]

print("\nAPERÇU DU DATASET ANALYTIQUE")
display(serial[preview_cols].head(15))

# --------------------------------------------
# 16. Exportation
# --------------------------------------------

serial.to_csv(
    OUTPUT / "serial_killers_analytical.csv",
    index=False
)

availability.to_csv(
    OUTPUT / "serial_features_availability.csv",
    index=False
)

print("\nEXPORTATION TERMINÉE")
print("Profils fusionnés :", len(serial))


### 8. Regroupement exploratoire des trajectoires criminelles

Cette étape vise à identifier des groupes statistiques parmi les tueurs en série documentés.

Trois caractéristiques quantitatives sont retenues : le nombre de victimes confirmées, la durée calendaire de l'activité criminelle et le nombre de territoires d'activité déclarés.

Les variables sont transformées et standardisées avant l'application de l'algorithme K-Means.

Plusieurs valeurs du nombre de groupes seront évaluées à l'aide du score de silhouette. La stabilité des résultats devra ensuite être vérifiée.

Les groupes obtenus représenteront des similarités statistiques entre trajectoires historiques. Ils ne constituent ni des diagnostics psychologiques ni des catégories permettant de prédire la dangerosité d'une personne.

In [ ]:

# ============================================
# SCI1402 - Étape 9
# Premier clustering exploratoire
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# --------------------------------------------
# 1. Sélection des variables
# --------------------------------------------

features = [
    "Proven victims",
    "Activity Years",
    "Country Count"
]

cluster_data = serial[
    ["Name"] + features
].copy()

# Conversion numérique
for col in features:
    cluster_data[col] = pd.to_numeric(
        cluster_data[col],
        errors="coerce"
    )

# Retirer les valeurs absentes ou incohérentes
cluster_data = cluster_data.dropna(
    subset=features
).copy()

cluster_data = cluster_data[
    (cluster_data[features] > 0).all(axis=1)
].copy()

print("PROFILS EXPLOITABLES")
print(len(cluster_data))

print("\nSTATISTIQUES DES VARIABLES")
display(cluster_data[features].describe())

# --------------------------------------------
# 2. Transformation logarithmique
# --------------------------------------------

X_log = np.log1p(
    cluster_data[features].astype(float)
)

# --------------------------------------------
# 3. Standardisation
# --------------------------------------------

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_log)

# --------------------------------------------
# 4. Évaluation du nombre de groupes
# --------------------------------------------

scores = []

for k in range(2, 9):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(X_scaled)

    score = silhouette_score(
        X_scaled,
        labels
    )

    scores.append({
        "Clusters": k,
        "Silhouette": score,
        "Inertie": model.inertia_
    })

scores_df = pd.DataFrame(scores)

print("\nÉVALUATION DES CLUSTERS")
display(scores_df)

# --------------------------------------------
# 5. Visualisation des scores
# --------------------------------------------

plt.figure(figsize=(10, 5))

sns.lineplot(
    data=scores_df,
    x="Clusters",
    y="Silhouette",
    marker="o"
)

plt.title(
    "Évaluation du nombre de groupes"
)

plt.xlabel("Nombre de clusters")
plt.ylabel("Score de silhouette")
plt.tight_layout()
plt.show()

# --------------------------------------------
# 6. Choix du meilleur K
# --------------------------------------------

best_k = int(
    scores_df.loc[
        scores_df["Silhouette"].idxmax(),
        "Clusters"
    ]
)

print("\nMEILLEUR K SELON LA SILHOUETTE")
print(best_k)

# --------------------------------------------
# 7. Modèle final exploratoire
# --------------------------------------------

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=20
)

cluster_data["Cluster"] = (
    kmeans.fit_predict(X_scaled)
)

print("\nTAILLE DES GROUPES")
display(
    cluster_data["Cluster"]
    .value_counts()
    .sort_index()
)

# --------------------------------------------
# 8. Description des groupes
# --------------------------------------------

summary = cluster_data.groupby(
    "Cluster"
)[features].agg(
    ["count", "median", "mean"]
)

print("\nDESCRIPTION DES GROUPES")
display(summary.round(2))

# --------------------------------------------
# 9. Projection PCA
# --------------------------------------------

pca = PCA(n_components=2)

coordinates = pca.fit_transform(
    X_scaled
)

cluster_data["PCA 1"] = coordinates[:, 0]
cluster_data["PCA 2"] = coordinates[:, 1]

print("\nVARIANCE EXPLIQUÉE PAR LA PCA")
print(
    np.round(
        pca.explained_variance_ratio_ * 100,
        2
    )
)

plt.figure(figsize=(11, 7))

sns.scatterplot(
    data=cluster_data,
    x="PCA 1",
    y="PCA 2",
    hue="Cluster",
    palette="tab10",
    alpha=0.7,
    s=45
)

plt.title(
    "Groupes de trajectoires criminelles"
)

plt.tight_layout()
plt.show()

# --------------------------------------------
# 10. Examiner les profils de chaque groupe
# --------------------------------------------

for cluster_id in sorted(
    cluster_data["Cluster"].unique()
):

    subset = cluster_data[
        cluster_data["Cluster"] == cluster_id
    ].sort_values(
        "Proven victims",
        ascending=False
    )

    print(
        f"\nCLUSTER {cluster_id} "
        f"({len(subset)} profils)"
    )

    display(
        subset[
            ["Name"] + features
        ].head(10)
    )

# --------------------------------------------
# 11. Exportation
# --------------------------------------------

cluster_data.to_csv(
    "/kaggle/working/"
    "serial_killers_clusters.csv",
    index=False
)

scores_df.to_csv(
    "/kaggle/working/"
    "clustering_scores.csv",
    index=False
)

print("\nCLUSTERING TERMINÉ")


### 9. Analyse de sensibilité du clustering

Le premier modèle K-Means a identifié deux groupes parmi 658 profils historiques, avec un score de silhouette de 0,594.

L'examen des caractéristiques des groupes suggère que la séparation est fortement influencée par le nombre de pays d'activité déclarés. Le premier groupe contient uniquement des individus associés à un pays, tandis que le second regroupe des individus associés à plusieurs pays.

Afin de déterminer si cette variable domine artificiellement la segmentation, trois configurations seront comparées.

Le modèle A conserve les trois variables initiales. Le modèle B retire la variable géographique. Le modèle C remplace cette variable par une fréquence moyenne annuelle calculée à partir des victimes confirmées et de la durée d'activité.

Pour chaque configuration, les variables seront transformées logarithmiquement, standardisées et soumises à K-Means. Les scores de silhouette, les tailles des groupes et les caractéristiques médianes seront comparés.

Cette analyse vise à évaluer la sensibilité des regroupements aux choix de variables, sans présumer qu'un score de silhouette élevé représente une meilleure typologie criminologique.

In [ ]:

# ============================================
# SCI1402 - Étape 10
# Analyse de sensibilité du clustering
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score
)

# --------------------------------------------
# 1. Préparer les données
# --------------------------------------------

output_dir = Path("/kaggle/working")
output_dir.mkdir(parents=True, exist_ok=True)

base_features = [
    "Proven victims",
    "Activity Years",
    "Country Count"
]

data = serial[
    ["Name"] + base_features
].copy()

for col in base_features:
    data[col] = pd.to_numeric(
        data[col],
        errors="coerce"
    )

data = data.dropna(
    subset=base_features
).copy()

data = data[
    (data[base_features] > 0).all(axis=1)
].copy()

data["Victims per Active Year"] = (
    data["Proven victims"]
    / data["Activity Years"]
)

print("PROFILS ANALYSÉS :", len(data))

# --------------------------------------------
# 2. Définir les configurations
# --------------------------------------------

configurations = {
    "A - Géographie": [
        "Proven victims",
        "Activity Years",
        "Country Count"
    ],
    "B - Sans géographie": [
        "Proven victims",
        "Activity Years"
    ],
    "C - Fréquence": [
        "Proven victims",
        "Activity Years",
        "Victims per Active Year"
    ]
}

# --------------------------------------------
# 3. Tester K = 2 à 8
# --------------------------------------------

all_scores = []
all_labels = {}
all_scaled = {}

for model_name, features in configurations.items():

    X = np.log1p(
        data[features].astype(float)
    )

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    all_scaled[model_name] = X_scaled

    for k in range(2, 9):

        model = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=20
        )

        labels = model.fit_predict(X_scaled)

        score = silhouette_score(
            X_scaled,
            labels
        )

        all_scores.append({
            "Model": model_name,
            "K": k,
            "Silhouette": score,
            "Inertia": model.inertia_
        })

        all_labels[(model_name, k)] = labels

scores_df = pd.DataFrame(all_scores)

# --------------------------------------------
# 4. Comparer les scores
# --------------------------------------------

print("\nSCORES DE SILHOUETTE")

pivot = scores_df.pivot(
    index="K",
    columns="Model",
    values="Silhouette"
)

display(pivot.round(4))

# --------------------------------------------
# 5. Graphique comparatif
# --------------------------------------------

plt.figure(figsize=(11, 6))

for model_name in configurations:

    subset = scores_df[
        scores_df["Model"] == model_name
    ]

    plt.plot(
        subset["K"],
        subset["Silhouette"],
        marker="o",
        label=model_name
    )

plt.title(
    "Sensibilité du clustering aux variables"
)

plt.xlabel("Nombre de clusters")
plt.ylabel("Score de silhouette")

plt.xticks(range(2, 9))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --------------------------------------------
# 6. Sélectionner le meilleur K
#    pour chaque configuration
# --------------------------------------------

best_models = {}

for model_name in configurations:

    subset = scores_df[
        scores_df["Model"] == model_name
    ]

    best_row = subset.loc[
        subset["Silhouette"].idxmax()
    ]

    best_k = int(best_row["K"])

    best_models[model_name] = {
        "K": best_k,
        "Silhouette": float(
            best_row["Silhouette"]
        ),
        "Labels": all_labels[
            (model_name, best_k)
        ]
    }

print("\nMEILLEURES CONFIGURATIONS")

best_summary = pd.DataFrame([
    {
        "Model": name,
        "Best K": result["K"],
        "Silhouette": result["Silhouette"]
    }
    for name, result in best_models.items()
])

display(best_summary.round(4))

# --------------------------------------------
# 7. Décrire les groupes
# --------------------------------------------

for model_name, result in best_models.items():

    labels = result["Labels"]

    temp = data.copy()
    temp["Cluster"] = labels

    print("\n" + "=" * 55)
    print(model_name)
    print("K =", result["K"])
    print("=" * 55)

    print("\nTAILLE DES GROUPES")
    display(
        temp["Cluster"]
        .value_counts()
        .sort_index()
    )

    print("\nMÉDIANES PAR GROUPE")
    display(
        temp.groupby("Cluster")[
            [
                "Proven victims",
                "Activity Years",
                "Country Count",
                "Victims per Active Year"
            ]
        ].median().round(2)
    )

# --------------------------------------------
# 8. Comparer les partitions avec
#    l'Adjusted Rand Index (ARI)
# --------------------------------------------

model_names = list(best_models.keys())

ari_matrix = pd.DataFrame(
    index=model_names,
    columns=model_names,
    dtype=float
)

for name_a in model_names:
    for name_b in model_names:

        ari_matrix.loc[name_a, name_b] = (
            adjusted_rand_score(
                best_models[name_a]["Labels"],
                best_models[name_b]["Labels"]
            )
        )

print("\nSIMILARITÉ ENTRE PARTITIONS (ARI)")
display(ari_matrix.round(4))

# --------------------------------------------
# 9. Tester explicitement l'association
#    du modèle A avec Country Count
# --------------------------------------------

data["Cluster A"] = (
    best_models["A - Géographie"]["Labels"]
)

data["Multi Country"] = (
    data["Country Count"] > 1
)

print("\nCLUSTER A VS PLUSIEURS PAYS")

cross_table = pd.crosstab(
    data["Cluster A"],
    data["Multi Country"]
)

display(cross_table)

# --------------------------------------------
# 10. Exporter les résultats
# --------------------------------------------

for model_name, result in best_models.items():

    safe_name = (
        model_name.split(" - ")[0]
    )

    data[f"Cluster {safe_name}"] = (
        result["Labels"]
    )

scores_df.to_csv(
    output_dir / "clustering_sensitivity_scores.csv",
    index=False
)

best_summary.to_csv(
    output_dir / "clustering_sensitivity_best.csv",
    index=False
)

ari_matrix.to_csv(
    output_dir / "clustering_sensitivity_ari.csv"
)

data.to_csv(
    output_dir / "clustering_sensitivity_profiles.csv",
    index=False
)

print("\nANALYSE DE SENSIBILITÉ TERMINÉE")


### 10. Validation de la stabilité du clustering

Le clustering exploratoire a mis en évidence plusieurs regroupements fondés sur le nombre de victimes confirmées et la durée d'activité criminelle. Les scores de silhouette obtenus pour trois, quatre et six groupes sont cependant très proches, ce qui nécessite une validation supplémentaire.

Cette étape vise à évaluer la robustesse des regroupements à l'aide de trois analyses complémentaires.

**Comparaison du nombre de groupes :** les partitions obtenues pour K = 3, 4 et 6 seront comparées selon leur silhouette, leurs effectifs et leurs caractéristiques statistiques.

**Stabilité par sous-échantillonnage :** trente échantillons représentant 80 % des profils seront tirés sans remise. Pour chaque échantillon, un nouveau modèle K-Means sera entraîné, puis ses affectations seront comparées à celles du modèle de référence sur les mêmes individus à l'aide de l'Adjusted Rand Index (ARI).

**Sensibilité aux valeurs extrêmes :** les modèles seront réentraînés après exclusion des observations dépassant le 99e percentile du nombre de victimes confirmées. Les affectations des profils conservés seront comparées aux affectations initiales.

Les transformations logarithmiques et la standardisation seront ajustées séparément sur chaque échantillon d'entraînement afin d'éviter toute fuite d'information entre les expériences.

L'objectif est de choisir une partition interprétable et suffisamment stable, sans considérer les groupes comme des catégories psychologiques ou des prédictions de comportements individuels.

In [ ]:

# ==================================================
# SCI1402 - ÉTAPE 11
# Validation de la stabilité du clustering
# ==================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score
)

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

RANDOM_STATE = 42
N_REPETITIONS = 30
SAMPLE_FRACTION = 0.80
K_VALUES = [3, 4, 6]

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = [
    "Proven victims",
    "Activity Years"
]

# --------------------------------------------------
# 2. Préparer les données
# --------------------------------------------------

data = serial[
    ["Name"] + FEATURES
].copy()

for col in FEATURES:
    data[col] = pd.to_numeric(
        data[col],
        errors="coerce"
    )

data = data.dropna(
    subset=FEATURES
).copy()

data = data[
    (data[FEATURES] > 0).all(axis=1)
].copy()

data = data.reset_index(drop=True)

print("PROFILS EXPLOITABLES :", len(data))

# --------------------------------------------------
# 3. Fonction de transformation
# --------------------------------------------------

def prepare_features(df, scaler=None):

    X_log = np.log1p(
        df[FEATURES].astype(float)
    )

    if scaler is None:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_log)
    else:
        X_scaled = scaler.transform(X_log)

    return X_scaled, scaler


# --------------------------------------------------
# 4. Fonction K-Means
# --------------------------------------------------

def fit_kmeans(df, k, random_state=42):

    X_scaled, scaler = prepare_features(df)

    model = KMeans(
        n_clusters=k,
        random_state=random_state,
        n_init=20
    )

    labels = model.fit_predict(X_scaled)

    return {
        "model": model,
        "scaler": scaler,
        "labels": labels,
        "X_scaled": X_scaled,
        "silhouette": silhouette_score(
            X_scaled,
            labels
        )
    }


# --------------------------------------------------
# 5. Modèles de référence
# --------------------------------------------------

reference_models = {}
reference_results = []

for k in K_VALUES:

    result = fit_kmeans(
        data,
        k,
        RANDOM_STATE
    )

    reference_models[k] = result

    sizes = pd.Series(
        result["labels"]
    ).value_counts()

    reference_results.append({
        "K": k,
        "Silhouette": result["silhouette"],
        "Smallest Cluster": sizes.min(),
        "Largest Cluster": sizes.max()
    })

reference_df = pd.DataFrame(
    reference_results
)

print("\nMODÈLES DE RÉFÉRENCE")
display(reference_df.round(4))

# --------------------------------------------------
# 6. Stabilité par sous-échantillonnage
# --------------------------------------------------

rng = np.random.default_rng(RANDOM_STATE)

stability_results = []

n_samples = int(
    len(data) * SAMPLE_FRACTION
)

for k in K_VALUES:

    reference_labels = (
        reference_models[k]["labels"]
    )

    for repetition in range(N_REPETITIONS):

        # Tirage sans remise
        indices = rng.choice(
            len(data),
            size=n_samples,
            replace=False
        )

        subset = data.iloc[indices].copy()

        # Nouveau scaler et nouveau modèle
        result = fit_kmeans(
            subset,
            k,
            RANDOM_STATE + repetition
        )

        # Comparaison sur les mêmes individus
        ari = adjusted_rand_score(
            reference_labels[indices],
            result["labels"]
        )

        stability_results.append({
            "K": k,
            "Repetition": repetition + 1,
            "ARI": ari,
            "Silhouette": result["silhouette"]
        })

stability_df = pd.DataFrame(
    stability_results
)

stability_summary = (
    stability_df.groupby("K")
    .agg(
        ARI_Mean=("ARI", "mean"),
        ARI_Median=("ARI", "median"),
        ARI_Std=("ARI", "std"),
        ARI_Min=("ARI", "min"),
        ARI_Max=("ARI", "max"),
        Silhouette_Mean=(
            "Silhouette", "mean"
        )
    )
    .reset_index()
)

print("\nSTABILITÉ PAR SOUS-ÉCHANTILLONNAGE")
display(stability_summary.round(4))

# --------------------------------------------------
# 7. Visualisation de la stabilité
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 6))

values = [
    stability_df.loc[
        stability_df["K"] == k,
        "ARI"
    ].to_numpy()
    for k in K_VALUES
]

ax.boxplot(
    values,
    tick_labels=[
        f"K={k}" for k in K_VALUES
    ],
    showmeans=True
)

ax.set_title(
    "Stabilité des partitions par sous-échantillonnage"
)

ax.set_ylabel(
    "Adjusted Rand Index (ARI)"
)

ax.set_xlabel(
    "Nombre de clusters"
)

ax.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()
plt.show()

# --------------------------------------------------
# 8. Sensibilité aux valeurs extrêmes
# --------------------------------------------------

threshold = data[
    "Proven victims"
].quantile(0.99)

mask = (
    data["Proven victims"] <= threshold
)

trimmed_data = data.loc[
    mask
].copy()

print("\nSEUIL DU 99e PERCENTILE")
print(threshold)

print("\nPROFILS CONSERVÉS")
print(len(trimmed_data))

print("\nPROFILS EXCLUS")
display(
    data.loc[
        ~mask,
        ["Name"] + FEATURES
    ].sort_values(
        "Proven victims",
        ascending=False
    )
)

# --------------------------------------------------
# 9. Comparer les modèles après exclusion
# --------------------------------------------------

outlier_results = []

for k in K_VALUES:

    original_labels = (
        reference_models[k]["labels"][mask.to_numpy()]
    )

    trimmed_result = fit_kmeans(
        trimmed_data,
        k,
        RANDOM_STATE
    )

    ari = adjusted_rand_score(
        original_labels,
        trimmed_result["labels"]
    )

    outlier_results.append({
        "K": k,
        "ARI After Trimming": ari,
        "Silhouette Original": (
            reference_models[k]["silhouette"]
        ),
        "Silhouette Trimmed": (
            trimmed_result["silhouette"]
        ),
        "Profiles Remaining": len(trimmed_data)
    })

outlier_df = pd.DataFrame(
    outlier_results
)

print("\nSENSIBILITÉ AUX VALEURS EXTRÊMES")
display(outlier_df.round(4))

# --------------------------------------------------
# 10. Comparer les partitions K=3,4,6
# --------------------------------------------------

comparison = pd.DataFrame(
    index=K_VALUES,
    columns=K_VALUES,
    dtype=float
)

for k1 in K_VALUES:
    for k2 in K_VALUES:

        comparison.loc[k1, k2] = (
            adjusted_rand_score(
                reference_models[k1]["labels"],
                reference_models[k2]["labels"]
            )
        )

print("\nCOMPARAISON ENTRE PARTITIONS")
display(comparison.round(4))

# --------------------------------------------------
# 11. Médianes par groupe
# --------------------------------------------------

for k in K_VALUES:

    temp = data.copy()

    temp["Cluster"] = (
        reference_models[k]["labels"]
    )

    print("\n" + "=" * 50)
    print(f"DESCRIPTION DES CLUSTERS - K={k}")
    print("=" * 50)

    summary = (
        temp.groupby("Cluster")
        .agg(
            Profiles=("Name", "count"),
            Victims_Median=(
                "Proven victims", "median"
            ),
            Activity_Median=(
                "Activity Years", "median"
            ),
            Victims_Mean=(
                "Proven victims", "mean"
            ),
            Activity_Mean=(
                "Activity Years", "mean"
            )
        )
    )

    display(summary.round(2))

# --------------------------------------------------
# 12. Synthèse finale
# --------------------------------------------------

final_summary = (
    reference_df
    .merge(
        stability_summary[
            [
                "K",
                "ARI_Mean",
                "ARI_Median",
                "ARI_Std"
            ]
        ],
        on="K"
    )
    .merge(
        outlier_df[
            [
                "K",
                "ARI After Trimming"
            ]
        ],
        on="K"
    )
)

print("\nSYNTHÈSE DE VALIDATION")
display(final_summary.round(4))

# --------------------------------------------------
# 13. Exportation
# --------------------------------------------------

reference_df.to_csv(
    OUTPUT_DIR / "validation_reference.csv",
    index=False
)

stability_df.to_csv(
    OUTPUT_DIR / "validation_stability_details.csv",
    index=False
)

stability_summary.to_csv(
    OUTPUT_DIR / "validation_stability_summary.csv",
    index=False
)

outlier_df.to_csv(
    OUTPUT_DIR / "validation_outliers.csv",
    index=False
)

comparison.to_csv(
    OUTPUT_DIR / "validation_partition_comparison.csv"
)

final_summary.to_csv(
    OUTPUT_DIR / "validation_final_summary.csv",
    index=False
)

print("\nVALIDATION TERMINÉE")


### 11. Exploration des descriptions textuelles

Le modèle K-Means à quatre groupes a été retenu comme représentation principale des trajectoires historiques, sur la base d'un compromis entre stabilité, simplicité et interprétabilité.

La présente étape examine le champ `Notes` du dataset *Wikipedia Serial Killers List* afin d'évaluer la possibilité d'en extraire des caractéristiques comportementales documentées.

Les descriptions seront associées aux quatre clusters, puis analysées selon leur disponibilité, leur longueur et leur vocabulaire.

Cette exploration ne cherchera pas encore à attribuer automatiquement des méthodes criminelles aux individus. Une absence de mention dans une description ne signifie pas qu'un comportement était absent.

Les résultats serviront à déterminer si une extraction structurée et vérifiable est réalisable pour la suite du projet.

In [ ]:

# ==================================================
# SCI1402 - ÉTAPE 12
# Exploration des descriptions criminologiques
# ==================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

from pathlib import Path
from collections import Counter
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 1. Reconstituer le modèle K = 4
# --------------------------------------------------

features = [
    "Proven victims",
    "Activity Years"
]

data = serial.copy()

for col in features:
    data[col] = pd.to_numeric(
        data[col],
        errors="coerce"
    )

data = data.dropna(
    subset=features
).copy()

data = data[
    (data[features] > 0).all(axis=1)
].copy()

X_log = np.log1p(
    data[features].astype(float)
)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

model = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=20
)

data["Cluster"] = model.fit_predict(X_scaled)

print("PROFILS ANALYSÉS :", len(data))

# --------------------------------------------------
# 2. Préparer les descriptions
# --------------------------------------------------

data["Notes"] = (
    data["Notes"]
    .astype("string")
    .str.strip()
)

data["Notes"] = data["Notes"].replace(
    ["", "Unknown", "Not available"],
    pd.NA
)

data["Notes Available"] = (
    data["Notes"].notna()
)

data["Notes Length"] = (
    data["Notes"].fillna("").str.len()
)

data["Word Count"] = (
    data["Notes"]
    .fillna("")
    .str.findall(r"\b\w+\b")
    .str.len()
)

# --------------------------------------------------
# 3. Disponibilité et longueur des textes
# --------------------------------------------------

summary = data.groupby("Cluster").agg(
    Profiles=("Name", "count"),
    Notes_Available=("Notes Available", "sum"),
    Median_Words=("Word Count", "median"),
    Mean_Words=("Word Count", "mean")
)

summary["Coverage (%)"] = (
    summary["Notes_Available"]
    / summary["Profiles"]
    * 100
)

print("\nQUALITÉ DES DESCRIPTIONS")
display(summary.round(2))

print("\nSTATISTIQUES GLOBALES")
display(
    data["Word Count"].describe()
)

# --------------------------------------------------
# 4. Distribution des longueurs
# --------------------------------------------------

plt.figure(figsize=(10, 5))

plt.hist(
    data.loc[
        data["Notes Available"],
        "Word Count"
    ],
    bins=30
)

plt.title("Longueur des descriptions")
plt.xlabel("Nombre de mots")
plt.ylabel("Nombre de profils")
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 5. Vocabulaire fréquent
# --------------------------------------------------

texts = (
    data.loc[
        data["Notes Available"],
        "Notes"
    ].astype(str)
)

vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=5,
    max_df=0.90,
    ngram_range=(1, 2),
    binary=True
)

if len(texts) > 0:

    X_text = vectorizer.fit_transform(texts)

    frequencies = np.asarray(
        X_text.sum(axis=0)
    ).ravel()

    vocabulary = pd.DataFrame({
        "Term": vectorizer.get_feature_names_out(),
        "Profiles Mentioning Term": frequencies
    })

    vocabulary = vocabulary.sort_values(
        "Profiles Mentioning Term",
        ascending=False
    )

    print("\nTERMES LES PLUS FRÉQUENTS")
    display(vocabulary.head(40))

else:
    vocabulary = pd.DataFrame(
        columns=[
            "Term",
            "Profiles Mentioning Term"
        ]
    )

    print("Aucune description disponible.")

# --------------------------------------------------
# 6. Exemples de descriptions par cluster
# --------------------------------------------------

for cluster_id in sorted(
    data["Cluster"].unique()
):

    subset = data[
        (data["Cluster"] == cluster_id)
        & data["Notes Available"]
    ].copy()

    # Échantillon reproductible
    sample = subset.sample(
        n=min(5, len(subset)),
        random_state=42
    ) if len(subset) else subset

    print("\n" + "=" * 55)
    print(f"CLUSTER {cluster_id}")
    print("=" * 55)

    for _, row in sample.iterrows():

        print("\nNOM :", row["Name"])
        print("VICTIMES :", row["Proven victims"])
        print("DURÉE :", row["Activity Years"])
        print("DESCRIPTION :")
        print(str(row["Notes"])[:1200])

# --------------------------------------------------
# 7. Exporter les résultats
# --------------------------------------------------

data.to_csv(
    OUTPUT_DIR / "serial_killers_k4_with_notes.csv",
    index=False
)

summary.to_csv(
    OUTPUT_DIR / "notes_quality_by_cluster.csv"
)

vocabulary.to_csv(
    OUTPUT_DIR / "notes_vocabulary.csv",
    index=False
)

print("\nEXPLORATION TEXTUELLE TERMINÉE")


### 12. Constitution d'un échantillon d'annotation comportementale

L'exploration textuelle a révélé une couverture de 99,85 % des descriptions, mais leur longueur médiane de 25 mots limite la quantité d'informations disponibles.

Afin de développer une méthode d'extraction reproductible, un échantillon stratifié de 80 dossiers sera constitué, avec 20 dossiers sélectionnés aléatoirement dans chacun des quatre clusters.

Chaque description sera examinée pour identifier les mentions explicites relatives aux méthodes criminelles, aux caractéristiques des victimes et aux contextes documentés.

Trois modalités seront distinguées : information explicitement présente, information explicitement absente et information inconnue ou non mentionnée. Les mentions incertaines, les actes attribués à des complices et les références à d'autres événements seront consignés séparément.

L'annotation manuelle servira de référence pour évaluer ultérieurement les performances d'une extraction automatique. Les résultats ne seront pas interprétés comme des descriptions exhaustives des comportements criminels.

In [ ]:

# ==================================================
# SCI1402 - ÉTAPE 13
# Échantillon d'annotation comportementale
# ==================================================

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

RANDOM_STATE = 42
SAMPLE_PER_CLUSTER = 20

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = [
    "Proven victims",
    "Activity Years"
]

# --------------------------------------------------
# 2. Reconstituer les quatre clusters
# --------------------------------------------------

data = serial.copy()

for col in FEATURES:
    data[col] = pd.to_numeric(
        data[col],
        errors="coerce"
    )

data = data.dropna(subset=FEATURES).copy()

data = data[
    (data[FEATURES] > 0).all(axis=1)
].copy()

data["Notes"] = (
    data["Notes"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

X = np.log1p(data[FEATURES].astype(float))
X_scaled = StandardScaler().fit_transform(X)

model = KMeans(
    n_clusters=4,
    random_state=RANDOM_STATE,
    n_init=20
)

data["Cluster"] = model.fit_predict(X_scaled)

# --------------------------------------------------
# 3. Sélectionner uniquement les descriptions
#    disponibles
# --------------------------------------------------

eligible = data[
    data["Notes"].notna()
].copy()

samples = []

for cluster_id in sorted(
    eligible["Cluster"].unique()
):

    subset = eligible[
        eligible["Cluster"] == cluster_id
    ]

    sample = subset.sample(
        n=min(SAMPLE_PER_CLUSTER, len(subset)),
        random_state=RANDOM_STATE
    )

    samples.append(sample)

annotation = pd.concat(
    samples,
    ignore_index=True
)

# --------------------------------------------------
# 4. Créer des identifiants d'annotation
# --------------------------------------------------

annotation = annotation.sort_values(
    ["Cluster", "Name"]
).reset_index(drop=True)

annotation.insert(
    0,
    "Annotation ID",
    [
        f"SCI1402-{i:03d}"
        for i in range(1, len(annotation) + 1)
    ]
)

# --------------------------------------------------
# 5. Définir les champs à annoter
# --------------------------------------------------

# Codes recommandés :
# 1 = explicitement présent
# 0 = explicitement absent
# vide = inconnu / non mentionné
#
# Ne jamais coder 0 simplement parce qu'un
# comportement n'est pas mentionné.

annotation_fields = [
    "Method Firearm",
    "Method Knife",
    "Method Strangulation",
    "Method Poisoning",
    "Method Blunt Force",
    "Method Other",
    "Victims Women",
    "Victims Men",
    "Victims Children",
    "Victims Elderly",
    "Victims Sex Workers",
    "Sexual Violence Mentioned",
    "Robbery Mentioned",
    "Dismemberment Mentioned",
    "Accomplices Mentioned"
]

for field in annotation_fields:
    annotation[field] = pd.Series(
        pd.NA,
        index=annotation.index,
        dtype="Int64"
    )

# --------------------------------------------------
# 6. Ajouter les champs de validation
# --------------------------------------------------

annotation["Record Type"] = pd.NA
annotation["Evidence Quote"] = pd.NA
annotation["Uncertainty Notes"] = pd.NA
annotation["Annotation Status"] = "Pending"

# Record Type :
# Individual / Group / Unidentified / Unclear
#
# Evidence Quote :
# Court extrait de Notes justifiant le codage
#
# Uncertainty Notes :
# Informations contestées, ambiguës,
# concernant des complices ou d'autres crimes
#
# Annotation Status :
# Pending / Reviewed

# --------------------------------------------------
# 7. Définir les colonnes du fichier final
# --------------------------------------------------

output_columns = [
    "Annotation ID",
    "Name",
    "Cluster",
    "Proven victims",
    "Activity Years",
    "Notes",
    "Record Type"
] + annotation_fields + [
    "Evidence Quote",
    "Uncertainty Notes",
    "Annotation Status"
]

annotation = annotation[output_columns]

# --------------------------------------------------
# 8. Vérifier la répartition
# --------------------------------------------------

print("ÉCHANTILLON TOTAL")
print(len(annotation))

print("\nRÉPARTITION PAR CLUSTER")
display(
    annotation["Cluster"]
    .value_counts()
    .sort_index()
)

print("\nAPERÇU DE LA GRILLE")
display(annotation.head(10))

# --------------------------------------------------
# 9. Exporter la grille d'annotation
# --------------------------------------------------

output_file = (
    OUTPUT_DIR
    / "serial_killers_manual_annotation.csv"
)

annotation.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("\nFICHIER CRÉÉ")
print(output_file)

print("\nÉCHANTILLON PRÊT POUR ANNOTATION")


### 13. Extraction automatique des caractéristiques criminologiques

Cette étape vise à développer un système d'extraction de mentions comportementales à partir des descriptions textuelles du dataset *Wikipedia Serial Killers List*.

Un échantillon stratifié de 80 dossiers, répartis également entre quatre clusters, est divisé en deux ensembles : 40 dossiers de développement et 40 dossiers de test.

L'extraction repose sur des expressions régulières associées à des catégories prédéfinies, notamment les méthodes criminelles, les caractéristiques des victimes et certains comportements connexes.

Chaque détection conserve une justification textuelle. Les expressions sont recherchées dans leur contexte immédiat afin de signaler certaines négations et formulations incertaines.

Les résultats automatiques sont stockés séparément des annotations manuelles. Une mention non détectée ne constitue pas une preuve d'absence du comportement.

Cette première approche constitue une base expérimentale. Elle nécessite une vérification humaine avant toute interprétation criminologique ou comparaison quantitative entre clusters.

In [ ]:

# ============================================================
# SCI1402 - ÉTAPE 14
# Extraction automatique de mentions criminologiques
# ============================================================

import re
import json
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

RANDOM_STATE = 42

OUTPUT_DIR = Path("/kaggle/working")

SOURCE_FILE = (
    OUTPUT_DIR / "serial_killers_manual_annotation.csv"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Charger la grille d'annotation
# ------------------------------------------------------------

if not SOURCE_FILE.exists():
    raise FileNotFoundError(
        f"Fichier introuvable : {SOURCE_FILE}"
    )

data = pd.read_csv(
    SOURCE_FILE,
    encoding="utf-8-sig",
    dtype={"Annotation ID": "string"}
)

required_columns = [
    "Annotation ID",
    "Name",
    "Cluster",
    "Notes",
    "Annotation Status"
]

missing_columns = [
    col for col in required_columns
    if col not in data.columns
]

if missing_columns:
    raise ValueError(
        f"Colonnes manquantes : {missing_columns}"
    )

if data["Annotation ID"].duplicated().any():
    raise ValueError(
        "Des identifiants d'annotation sont dupliqués."
    )

if len(data) != 80:
    raise ValueError(
        f"80 dossiers attendus, {len(data)} trouvés."
    )

data["Cluster"] = pd.to_numeric(
    data["Cluster"],
    errors="raise"
).astype(int)

data["Notes"] = (
    data["Notes"]
    .astype("string")
    .fillna("")
)

print("DOSSIERS CHARGÉS :", len(data))

# ------------------------------------------------------------
# 3. Séparation stratifiée développement / test
# ------------------------------------------------------------

development, test = train_test_split(
    data,
    test_size=0.50,
    stratify=data["Cluster"],
    random_state=RANDOM_STATE
)

development = development.copy().sort_values(
    "Annotation ID"
).reset_index(drop=True)

test = test.copy().sort_values(
    "Annotation ID"
).reset_index(drop=True)

# Vérifier qu'aucun dossier n'est partagé
overlap = set(
    development["Annotation ID"]
).intersection(
    set(test["Annotation ID"])
)

assert len(overlap) == 0

print("\nRÉPARTITION DES ÉCHANTILLONS")

split_summary = pd.DataFrame({
    "Development": development[
        "Cluster"
    ].value_counts().sort_index(),
    "Test": test[
        "Cluster"
    ].value_counts().sort_index()
})

display(split_summary)

development.to_csv(
    OUTPUT_DIR / "annotation_development.csv",
    index=False,
    encoding="utf-8-sig"
)

test.to_csv(
    OUTPUT_DIR / "annotation_test.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 4. Dictionnaire de détection
# ------------------------------------------------------------
#
# Les expressions repèrent des mentions candidates.
#
# Une détection ne confirme pas :
# - que le comportement est établi;
# - que le comportement concerne l'auteur principal;
# - que toutes les victimes sont concernées.
#
# Les expressions sont volontairement limitées.
# Elles seront améliorées sur l'ensemble de développement.
# ------------------------------------------------------------

PATTERNS = {

    "Method Firearm": [
        r"\b(?:shot|shooting|shootings)\b",
        r"\b(?:gun|guns|gunshot|gunshots)\b",
        r"\b(?:firearm|firearms)\b",
        r"\b(?:rifle|rifles|revolver|revolvers)\b",
        r"\b(?:pistol|pistols)\b"
    ],

    "Method Knife": [
        r"\b(?:stab|stabs|stabbed|stabbing)\b",
        r"\b(?:knife|knives)\b",
        r"\b(?:slashed|slashing)\b"
    ],

    "Method Strangulation": [
        r"\b(?:strangle|strangled|strangling)\b",
        r"\b(?:strangulation)\b",
        r"\b(?:choke|choked|choking)\b",
        r"\b(?:suffocate|suffocated|suffocation)\b"
    ],

    "Method Poisoning": [
        r"\b(?:poison|poisoned|poisoning|poisoner)\b",
        r"\b(?:arsenic|cyanide)\b",
        r"\b(?:overdose|overdoses)\b"
    ],

    "Method Blunt Force": [
        r"\b(?:bludgeon|bludgeoned|bludgeoning)\b",
        r"\b(?:beaten|beat to death)\b",
        r"\b(?:blunt force)\b",
        r"\b(?:hammer|clubbed)\b"
    ],

    "Method Other": [
        r"\b(?:drowned|drowning)\b",
        r"\b(?:burned alive|burnt alive)\b",
        r"\b(?:smothered|smothering)\b"
    ],

    "Victims Women": [
        r"\b(?:women|woman)\b",
        r"\b(?:female victims)\b",
        r"\b(?:girls|girl)\b"
    ],

    "Victims Men": [
        r"\b(?:men|man)\b",
        r"\b(?:male victims)\b",
        r"\b(?:boys|boy)\b"
    ],

    "Victims Children": [
        r"\b(?:children|child)\b",
        r"\b(?:babies|baby|infants|infant)\b",
        r"\b(?:boys|girls)\b",
        r"\b(?:juveniles|juvenile)\b"
    ],

    "Victims Elderly": [
        r"\b(?:elderly|seniors)\b",
        r"\b(?:old age pensioners)\b",
        r"\b(?:elderly residents)\b"
    ],

    "Victims Sex Workers": [
        r"\b(?:sex workers?|prostitutes?)\b",
        r"\b(?:sex trade workers?)\b"
    ],

    "Sexual Violence Mentioned": [
        r"\b(?:rape|raped|raping|rapist)\b",
        r"\b(?:sexual assault|sexual assaults)\b",
        r"\b(?:sexual violence|sexual abuse)\b"
    ],

    "Robbery Mentioned": [
        r"\b(?:robbery|robberies|robbed)\b",
        r"\b(?:burglaries|burglary|burglar)\b"
    ],

    "Dismemberment Mentioned": [
        r"\b(?:dismembered|dismemberment)\b",
        r"\b(?:mutilated|mutilation)\b"
    ],

    "Accomplices Mentioned": [
        r"\b(?:accomplice|accomplices)\b",
        r"\b(?:partner in crime)\b",
        r"\b(?:with the help of)\b",
        r"\b(?:co-conspirator|co-conspirators)\b"
    ]
}

COMPILED_PATTERNS = {
    category: [
        re.compile(pattern, re.IGNORECASE)
        for pattern in patterns
    ]
    for category, patterns in PATTERNS.items()
}

# ------------------------------------------------------------
# 5. Détection de contexte
# ------------------------------------------------------------
#
# Ces indicateurs sont des avertissements heuristiques.
# Ils ne résolvent pas automatiquement les ambiguïtés.
# ------------------------------------------------------------

NEGATION_PATTERN = re.compile(
    r"\b(?:no|not|never|without|denied|"
    r"didn't|did not|wasn't|were not)\b",
    re.IGNORECASE
)

UNCERTAINTY_PATTERN = re.compile(
    r"\b(?:suspected|alleged|allegedly|"
    r"possibly|possibly linked|may have|"
    r"might have|believed|could have|"
    r"accused|reportedly)\b",
    re.IGNORECASE
)

OTHER_PERSON_PATTERN = re.compile(
    r"\b(?:father|mother|brother|sister|"
    r"accomplice|partner|police officer|"
    r"another killer|other suspect)\b",
    re.IGNORECASE
)

# ------------------------------------------------------------
# 6. Fonction d'extraction
# ------------------------------------------------------------

def extract_mentions(text, context_chars=90):

    text = str(text) if pd.notna(text) else ""

    results = {}

    for category, regex_list in COMPILED_PATTERNS.items():

        matches = []

        for regex in regex_list:

            for match in regex.finditer(text):

                start = match.start()
                end = match.end()

                left = max(
                    0,
                    start - context_chars
                )

                right = min(
                    len(text),
                    end + context_chars
                )

                snippet = text[left:right].strip()

                # Fenêtre précédente utilisée uniquement
                # comme signal d'alerte contextuel.
                before = text[
                    max(0, start - 60):start
                ]

                context = text[left:right]

                flags = []

                if NEGATION_PATTERN.search(before):
                    flags.append("possible_negation")

                if UNCERTAINTY_PATTERN.search(context):
                    flags.append("possible_uncertainty")

                if OTHER_PERSON_PATTERN.search(context):
                    flags.append("possible_other_person")

                matches.append({
                    "term": match.group(),
                    "start": start,
                    "end": end,
                    "evidence": snippet,
                    "flags": flags
                })

        # Retirer les doublons dus aux expressions
        # qui reconnaissent le même segment.
        unique_matches = {
            (
                item["start"],
                item["end"]
            ): item
            for item in matches
        }

        matches = sorted(
            unique_matches.values(),
            key=lambda item: (
                item["start"],
                item["end"]
            )
        )

        results[category] = {
            "detected": len(matches) > 0,
            "matches": matches
        }

    return results


# ------------------------------------------------------------
# 7. Appliquer l'extracteur
# ------------------------------------------------------------

def build_predictions(df):

    records = []

    for _, row in df.iterrows():

        extracted = extract_mentions(
            row["Notes"]
        )

        record = {
            "Annotation ID": row["Annotation ID"],
            "Name": row["Name"],
            "Cluster": row["Cluster"],
            "Notes": row["Notes"]
        }

        for category, result in extracted.items():

            # 1 = mention candidate détectée
            # NA = aucune mention détectée
            #
            # Nous n'utilisons pas 0 comme preuve
            # d'absence d'un comportement.

            record[category + " Predicted"] = (
                1 if result["detected"] else pd.NA
            )

            record[category + " Evidence"] = (
                json.dumps(
                    result["matches"],
                    ensure_ascii=False
                )
                if result["detected"]
                else ""
            )

        records.append(record)

    return pd.DataFrame(records)


development_predictions = build_predictions(
    development
)

test_predictions = build_predictions(
    test
)

# ------------------------------------------------------------
# 8. Résumer les détections sur le développement
# ------------------------------------------------------------

summary_rows = []

for category in PATTERNS:

    col = category + " Predicted"

    detected = int(
        development_predictions[col]
        .notna()
        .sum()
    )

    summary_rows.append({
        "Category": category,
        "Detected Profiles": detected,
        "Development Profiles": len(development),
        "Detection Rate (%)": (
            detected / len(development) * 100
        )
    })

summary = pd.DataFrame(
    summary_rows
).sort_values(
    "Detected Profiles",
    ascending=False
)

print("\nMENTIONS DÉTECTÉES - DÉVELOPPEMENT")
display(summary.round(2))

# ------------------------------------------------------------
# 9. Exemples de justifications
# ------------------------------------------------------------

print("\nEXEMPLES DE DÉTECTIONS")

examples_shown = 0

for _, row in development_predictions.iterrows():

    for category in PATTERNS:

        predicted_col = category + " Predicted"
        evidence_col = category + " Evidence"

        if pd.notna(row[predicted_col]):

            evidence = json.loads(
                row[evidence_col]
            )

            if not evidence:
                continue

            first_match = evidence[0]

            print("\nDOSSIER :", row["Annotation ID"])
            print("NOM :", row["Name"])
            print("CATÉGORIE :", category)
            print("TERME :", first_match["term"])
            print("EXTRAIT :", first_match["evidence"])
            print("ALERTES :", first_match["flags"])

            examples_shown += 1

            if examples_shown >= 12:
                break

    if examples_shown >= 12:
        break

# ------------------------------------------------------------
# 10. Exportation
# ------------------------------------------------------------

development_predictions.to_csv(
    OUTPUT_DIR / "nlp_predictions_development.csv",
    index=False,
    encoding="utf-8-sig"
)

test_predictions.to_csv(
    OUTPUT_DIR / "nlp_predictions_test.csv",
    index=False,
    encoding="utf-8-sig"
)

summary.to_csv(
    OUTPUT_DIR / "nlp_extraction_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nFICHIERS EXPORTÉS")
print("annotation_development.csv")
print("annotation_test.csv")
print("nlp_predictions_development.csv")
print("nlp_predictions_test.csv")
print("nlp_extraction_summary.csv")

print("\nÉTAPE 14 TERMINÉE")


### 14. Préparation de la validation humaine des extractions

Les premières règles d'extraction ont permis d'identifier des mentions relatives aux méthodes criminelles, aux caractéristiques des victimes et aux comportements associés.

Afin de mesurer leur fiabilité, une grille de validation humaine est préparée pour les 40 dossiers de développement.

Chaque catégorie fera l'objet d'une annotation binaire : une valeur de 1 indique qu'une mention pertinente est explicitement présente dans le texte, tandis qu'une valeur de 0 indique qu'aucune mention pertinente n'y figure. Une cellule vide correspond à une annotation non réalisée.

Ces annotations concernent uniquement le contenu documentaire et ne constituent pas des affirmations sur l'absence réelle d'un comportement.

Les annotations humaines seront conservées séparément des prédictions automatiques. Une fois complétées, elles permettront de calculer la précision, le rappel et le score F1 pour chaque catégorie disposant de suffisamment d'exemples.

In [ ]:

# ============================================================
# SCI1402 - ÉTAPE 15
# Préparation de la validation humaine du NLP
# ============================================================

import json
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working")

ANNOTATION_FILE = (
    OUTPUT_DIR / "annotation_development.csv"
)

PREDICTIONS_FILE = (
    OUTPUT_DIR / "nlp_predictions_development.csv"
)

REVIEW_FILE = (
    OUTPUT_DIR / "nlp_manual_review_development.csv"
)

# ------------------------------------------------------------
# 1. Vérifier les fichiers
# ------------------------------------------------------------

for file in [ANNOTATION_FILE, PREDICTIONS_FILE]:
    if not file.exists():
        raise FileNotFoundError(
            f"Fichier introuvable : {file}"
        )

annotations = pd.read_csv(
    ANNOTATION_FILE,
    encoding="utf-8-sig",
    dtype={"Annotation ID": "string"}
)

predictions = pd.read_csv(
    PREDICTIONS_FILE,
    encoding="utf-8-sig",
    dtype={"Annotation ID": "string"}
)

if annotations["Annotation ID"].duplicated().any():
    raise ValueError(
        "Identifiants dupliqués dans les annotations."
    )

if predictions["Annotation ID"].duplicated().any():
    raise ValueError(
        "Identifiants dupliqués dans les prédictions."
    )

# ------------------------------------------------------------
# 2. Définir les catégories
# ------------------------------------------------------------

categories = [
    "Method Firearm",
    "Method Knife",
    "Method Strangulation",
    "Method Poisoning",
    "Method Blunt Force",
    "Method Other",
    "Victims Women",
    "Victims Men",
    "Victims Children",
    "Victims Elderly",
    "Victims Sex Workers",
    "Sexual Violence Mentioned",
    "Robbery Mentioned",
    "Dismemberment Mentioned",
    "Accomplices Mentioned"
]

# ------------------------------------------------------------
# 3. Fusionner par identifiant
# ------------------------------------------------------------

prediction_columns = ["Annotation ID"]

for category in categories:
    prediction_columns.extend([
        category + " Predicted",
        category + " Evidence"
    ])

missing = [
    col for col in prediction_columns
    if col not in predictions.columns
]

if missing:
    raise ValueError(
        f"Colonnes manquantes : {missing}"
    )

review = annotations[
    [
        "Annotation ID",
        "Name",
        "Cluster",
        "Notes"
    ]
].merge(
    predictions[prediction_columns],
    on="Annotation ID",
    how="left",
    validate="one_to_one",
    indicator=True
)

if not (review["_merge"] == "both").all():
    raise ValueError(
        "Certaines prédictions sont manquantes."
    )

review = review.drop(columns="_merge")

# ------------------------------------------------------------
# 4. Créer une grille longue
#    Une ligne = un dossier et une catégorie
# ------------------------------------------------------------

records = []

for _, row in review.iterrows():

    for category in categories:

        predicted_col = category + " Predicted"
        evidence_col = category + " Evidence"

        raw_prediction = row[predicted_col]

        # Pour l'évaluation de la détection textuelle :
        # 1 = mention candidate trouvée
        # 0 = aucune mention candidate trouvée

        predicted = (
            1 if pd.notna(raw_prediction)
            else 0
        )

        evidence = row[evidence_col]

        if pd.isna(evidence):
            evidence = ""

        try:
            matches = (
                json.loads(evidence)
                if evidence else []
            )
        except json.JSONDecodeError:
            matches = []

        snippets = [
            match.get("evidence", "")
            for match in matches
        ]

        flags = sorted({
            flag
            for match in matches
            for flag in match.get("flags", [])
        })

        records.append({
            "Annotation ID": row["Annotation ID"],
            "Name": row["Name"],
            "Cluster": row["Cluster"],
            "Category": category,
            "Notes": row["Notes"],
            "Predicted": predicted,
            "Detected Evidence": " | ".join(snippets),
            "Context Flags": " | ".join(flags),
            "Human Label": pd.NA,
            "Human Evidence": pd.NA,
            "Review Notes": pd.NA,
            "Review Status": "Pending"
        })

review_long = pd.DataFrame(records)

# ------------------------------------------------------------
# 5. Vérifications
# ------------------------------------------------------------

expected_rows = len(annotations) * len(categories)

assert len(review_long) == expected_rows

assert review_long[
    ["Annotation ID", "Category"]
].duplicated().sum() == 0

print("DOSSIERS :", len(annotations))
print("CATÉGORIES :", len(categories))
print("ANNOTATIONS À VÉRIFIER :", len(review_long))

print("\nPRÉDICTIONS POSITIVES")
print(int(review_long["Predicted"].sum()))

print("\nPRÉDICTIONS NÉGATIVES")
print(int((review_long["Predicted"] == 0).sum()))

# ------------------------------------------------------------
# 6. Aperçu
# ------------------------------------------------------------

print("\nAPERÇU DE LA GRILLE")

display(
    review_long[
        [
            "Annotation ID",
            "Category",
            "Predicted",
            "Detected Evidence",
            "Human Label",
            "Review Status"
        ]
    ].head(20)
)

# ------------------------------------------------------------
# 7. Exporter
# ------------------------------------------------------------

review_long.to_csv(
    REVIEW_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nGRILLE CRÉÉE")
print(REVIEW_FILE)

print("\nÉTAPE 15 - PRÉPARATION TERMINÉE")


### 15. Interface interactive de validation humaine

Une interface interactive est développée avec `ipywidgets` afin de faciliter la validation des mentions criminologiques extraites automatiquement.

Les 40 dossiers de développement sont présentés individuellement avec leur description originale, les catégories d'analyse et les propositions de l'extracteur.

L'annotateur peut indiquer si une catégorie est explicitement mentionnée, absente du texte ou encore à vérifier. Il peut également conserver une justification textuelle et des commentaires sur les ambiguïtés.

Les annotations sont sauvegardées dans un fichier CSV et peuvent être reprises après une interruption de la session, à condition que le fichier ait été conservé.

Cette procédure vise à constituer un ensemble de référence humain permettant de mesurer ultérieurement la précision, le rappel et le score F1 de l'extracteur. Les annotations portent sur les mentions documentaires et non sur la présence ou l'absence réelle des comportements étudiés.

In [ ]:

# ============================================================
# SCI1402 - ÉTAPE 16
# Interface interactive d'annotation humaine
# ============================================================

import shutil
import html
from pathlib import Path
from datetime import datetime

import pandas as pd
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

OUTPUT_DIR = Path("/kaggle/working")

REVIEW_FILE = (
    OUTPUT_DIR / "nlp_manual_review_development.csv"
)

BACKUP_FILE = (
    OUTPUT_DIR / "nlp_manual_review_development_backup.csv"
)

REQUIRED_COLUMNS = [
    "Annotation ID",
    "Name",
    "Cluster",
    "Category",
    "Notes",
    "Predicted",
    "Human Label",
    "Human Evidence",
    "Review Notes",
    "Review Status"
]

if not REVIEW_FILE.exists():
    raise FileNotFoundError(
        f"Fichier introuvable : {REVIEW_FILE}"
    )

# ------------------------------------------------------------
# 2. Charger les annotations existantes
# ------------------------------------------------------------

df = pd.read_csv(
    REVIEW_FILE,
    encoding="utf-8-sig",
    dtype={"Annotation ID": "string"}
)

missing = [
    col for col in REQUIRED_COLUMNS
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Colonnes manquantes : {missing}"
    )

if df[["Annotation ID", "Category"]].duplicated().any():
    raise ValueError(
        "Des annotations sont dupliquées."
    )

# Normalisation des colonnes
for col in ["Human Evidence", "Review Notes"]:
    df[col] = df[col].astype("string")

df["Human Label"] = pd.to_numeric(
    df["Human Label"],
    errors="coerce"
).astype("Int64")

df["Predicted"] = pd.to_numeric(
    df["Predicted"],
    errors="coerce"
).astype("Int64")

df["Review Status"] = (
    df["Review Status"]
    .fillna("Pending")
    .astype(str)
)

# Vérifier les annotations existantes
valid_labels = df["Human Label"].isna() | (
    df["Human Label"].isin([0, 1])
)

if not valid_labels.all():
    raise ValueError(
        "Human Label doit contenir 0, 1 ou une valeur vide."
    )

# Conserver une copie de sécurité au lancement
shutil.copy2(REVIEW_FILE, BACKUP_FILE)

# ------------------------------------------------------------
# 3. Identifier les dossiers
# ------------------------------------------------------------

case_ids = sorted(
    df["Annotation ID"].dropna().unique().tolist()
)

categories = df["Category"].dropna().unique().tolist()

TOTAL_CASES = len(case_ids)
TOTAL_CATEGORIES = len(categories)

if TOTAL_CASES != 40 or TOTAL_CATEGORIES != 15:
    raise ValueError(
        "La grille attendue doit contenir "
        "40 dossiers et 15 catégories."
    )

# Vérifier la complétude de la grille
for case_id in case_ids:
    subset = df[df["Annotation ID"] == case_id]
    if len(subset) != TOTAL_CATEGORIES:
        raise ValueError(
            f"Grille incomplète pour {case_id}"
        )

# ------------------------------------------------------------
# 4. Descriptions françaises
# ------------------------------------------------------------

CATEGORY_LABELS = {
    "Method Firearm": "Arme à feu",
    "Method Knife": "Arme blanche",
    "Method Strangulation": "Étranglement",
    "Method Poisoning": "Empoisonnement",
    "Method Blunt Force": "Objet contondant",
    "Method Other": "Autre méthode",
    "Victims Women": "Femmes / filles",
    "Victims Men": "Hommes / garçons",
    "Victims Children": "Enfants / mineurs",
    "Victims Elderly": "Personnes âgées",
    "Victims Sex Workers": "Travailleurs du sexe",
    "Sexual Violence Mentioned": "Violence sexuelle",
    "Robbery Mentioned": "Vol / cambriolage",
    "Dismemberment Mentioned": "Démembrement / mutilation",
    "Accomplices Mentioned": "Complices"
}

LABEL_OPTIONS = [
    ("À vérifier", "unknown"),
    ("Oui — mention pertinente", "1"),
    ("Non — aucune mention pertinente", "0")
]

# ------------------------------------------------------------
# 5. État de l'interface
# ------------------------------------------------------------

state = {
    "case_index": 0,
    "loading": False,
    "dirty": False
}

category_controls = {}

# ------------------------------------------------------------
# 6. Composants principaux
# ------------------------------------------------------------

title = widgets.HTML()

progress = widgets.HTML()

description = widgets.HTML()

status_message = widgets.HTML()

case_selector = widgets.Dropdown(
    options=[
        (case_id, case_id)
        for case_id in case_ids
    ],
    description="Dossier :",
    layout=widgets.Layout(width="280px"),
    style={"description_width": "initial"}
)

previous_button = widgets.Button(
    description="Précédent",
    icon="arrow-left",
    button_style=""
)

next_button = widgets.Button(
    description="Suivant",
    icon="arrow-right",
    button_style=""
)

save_button = widgets.Button(
    description="Enregistrer",
    icon="save",
    button_style="success"
)

show_predictions = widgets.Checkbox(
    value=False,
    description="Afficher les prédictions automatiques",
    indent=False
)

show_only_pending = widgets.Checkbox(
    value=False,
    description="Afficher uniquement les catégories non validées",
    indent=False
)

mark_all_zero_button = widgets.Button(
    description="Mettre les catégories restantes à Non",
    button_style="warning",
    icon="check-square",
    tooltip="À utiliser seulement après lecture complète du texte"
)

category_container = widgets.VBox()

# ------------------------------------------------------------
# 7. Construire les contrôles des catégories
# ------------------------------------------------------------

for category in categories:

    label = widgets.HTML(
        value=(
            "<b>" +
            html.escape(
                CATEGORY_LABELS.get(category, category)
            ) +
            "</b>"
        )
    )

    selection = widgets.ToggleButtons(
        options=LABEL_OPTIONS,
        value="unknown",
        style={"button_width": "auto"},
        layout=widgets.Layout(width="100%")
    )

    evidence = widgets.Textarea(
        value="",
        placeholder=(
            "Extrait exact justifiant la mention "
            "(si Oui)"
        ),
        description="Preuve :",
        layout=widgets.Layout(width="100%", height="70px"),
        style={"description_width": "75px"}
    )

    notes = widgets.Textarea(
        value="",
        placeholder=(
            "Ambiguïté, négation, autre personne, "
            "formulation incertaine..."
        ),
        description="Notes :",
        layout=widgets.Layout(width="100%", height="65px"),
        style={"description_width": "75px"}
    )

    prediction = widgets.HTML()

    panel = widgets.VBox(
        [
            label,
            selection,
            prediction,
            evidence,
            notes
        ],
        layout=widgets.Layout(
            border="1px solid #cccccc",
            padding="10px",
            margin="4px 0"
        )
    )

    category_controls[category] = {
        "selection": selection,
        "evidence": evidence,
        "notes": notes,
        "prediction": prediction,
        "panel": panel
    }

# ------------------------------------------------------------
# 8. Sauvegarde
# ------------------------------------------------------------

def save_current_case():

    case_id = case_ids[state["case_index"]]

    for category, controls in category_controls.items():

        mask = (
            (df["Annotation ID"] == case_id) &
            (df["Category"] == category)
        )

        value = controls["selection"].value

        if value == "unknown":
            human_label = pd.NA
            review_status = "Pending"
        else:
            human_label = int(value)
            review_status = "Reviewed"

        evidence = controls["evidence"].value.strip()
        notes = controls["notes"].value.strip()

        df.loc[mask, "Human Label"] = human_label
        df.loc[mask, "Review Status"] = review_status

        df.loc[mask, "Human Evidence"] = (
            evidence if evidence else pd.NA
        )

        df.loc[mask, "Review Notes"] = (
            notes if notes else pd.NA
        )

    # Écriture dans un fichier temporaire,
    # puis remplacement du fichier principal.
    temp_file = OUTPUT_DIR / (
        "nlp_manual_review_development.tmp"
    )

    df.to_csv(
        temp_file,
        index=False,
        encoding="utf-8-sig"
    )

    temp_file.replace(REVIEW_FILE)

    state["dirty"] = False

    status_message.value = (
        "<span style='color:green'>"
        "Annotations enregistrées."
        "</span>"
    )

    update_progress()

# ------------------------------------------------------------
# 9. Barre de progression
# ------------------------------------------------------------

def update_progress():

    completed = int(
        (df["Review Status"] == "Reviewed").sum()
    )

    total = len(df)

    percentage = (
        completed / total * 100
        if total else 0
    )

    completed_cases = 0

    for case_id in case_ids:

        subset = df[
            df["Annotation ID"] == case_id
        ]

        if (
            subset["Review Status"] == "Reviewed"
        ).all():
            completed_cases += 1

    progress.value = f"""
    <div style="padding:10px 0">
      <b>Progression :</b>
      {completed} / {total} annotations
      ({percentage:.1f} %)
      <br>
      <b>Dossiers complets :</b>
      {completed_cases} / {TOTAL_CASES}
      <div style="
          background:#ddd;
          height:14px;
          border-radius:7px;
          margin-top:8px;
          overflow:hidden;
      ">
        <div style="
            background:#378a59;
            width:{percentage:.1f}%;
            height:14px;
        "></div>
      </div>
    </div>
    """

# ------------------------------------------------------------
# 10. Afficher les prédictions
# ------------------------------------------------------------

def update_prediction_visibility():

    case_id = case_ids[state["case_index"]]

    for category, controls in category_controls.items():

        mask = (
            (df["Annotation ID"] == case_id) &
            (df["Category"] == category)
        )

        row = df.loc[mask].iloc[0]

        if not show_predictions.value:
            controls["prediction"].value = ""
            continue

        predicted = row["Predicted"]

        if pd.isna(predicted):
            predicted_text = "Indéterminé"
        elif int(predicted) == 1:
            predicted_text = "Mention candidate détectée"
        else:
            predicted_text = "Aucune mention détectée"

        evidence = row.get("Detected Evidence", "")

        if pd.isna(evidence):
            evidence = ""

        controls["prediction"].value = (
            "<div style='padding:5px;color:#777'>"
            "<b>Extracteur :</b> " +
            html.escape(predicted_text) +
            "<br>" +
            html.escape(str(evidence)) +
            "</div>"
        )

# ------------------------------------------------------------
# 11. Filtrer les catégories
# ------------------------------------------------------------

def update_category_visibility():

    visible_panels = []

    for category, controls in category_controls.items():

        if (
            show_only_pending.value
            and controls["selection"].value != "unknown"
        ):
            continue

        visible_panels.append(
            controls["panel"]
        )

    category_container.children = tuple(
        visible_panels
    )

# ------------------------------------------------------------
# 12. Charger un dossier
# ------------------------------------------------------------

def load_case(index):

    state["loading"] = True

    try:

        state["case_index"] = index

        case_id = case_ids[index]

        subset = df[
            df["Annotation ID"] == case_id
        ]

        first_row = subset.iloc[0]

        name = html.escape(
            str(first_row["Name"])
        )

        cluster = first_row["Cluster"]

        text = html.escape(
            str(first_row["Notes"])
        )

        title.value = f"""
        <h3>
          Dossier {index + 1} / {TOTAL_CASES}
          — {name}
        </h3>
        <p>
          <b>ID :</b> {html.escape(case_id)}
          &nbsp;&nbsp;
          <b>Cluster :</b> {cluster}
        </p>
        """

        description.value = f"""
        <div style="
            border:1px solid #ccc;
            padding:16px;
            margin:10px 0;
            border-radius:6px;
            white-space:pre-wrap;
            line-height:1.6;
        ">
          <b>Description originale :</b>
          <p>{text}</p>
        </div>
        """

        for category, controls in category_controls.items():

            row = subset[
                subset["Category"] == category
            ].iloc[0]

            human_label = row["Human Label"]

            if pd.isna(human_label):
                value = "unknown"
            else:
                value = str(int(human_label))

            controls["selection"].value = value

            evidence = row["Human Evidence"]
            notes = row["Review Notes"]

            controls["evidence"].value = (
                "" if pd.isna(evidence)
                else str(evidence)
            )

            controls["notes"].value = (
                "" if pd.isna(notes)
                else str(notes)
            )

        case_selector.value = case_id

        previous_button.disabled = index == 0

        next_button.disabled = (
            index == TOTAL_CASES - 1
        )

        status_message.value = ""

        state["dirty"] = False

        update_prediction_visibility()
        update_category_visibility()
        update_progress()

    finally:
        state["loading"] = False

# ------------------------------------------------------------
# 13. Suivi des modifications
# ------------------------------------------------------------

def on_field_change(change):

    if state["loading"]:
        return

    state["dirty"] = True

    status_message.value = (
        "<span style='color:#b07000'>"
        "Modifications non enregistrées."
        "</span>"
    )

    update_category_visibility()


for controls in category_controls.values():

    controls["selection"].observe(
        on_field_change,
        names="value"
    )

    controls["evidence"].observe(
        on_field_change,
        names="value"
    )

    controls["notes"].observe(
        on_field_change,
        names="value"
    )

# ------------------------------------------------------------
# 14. Navigation
# ------------------------------------------------------------

def go_to_case(index):

    if index < 0 or index >= TOTAL_CASES:
        return

    # Enregistrement avant changement de dossier
    if state["dirty"]:
        save_current_case()

    load_case(index)


def on_previous_clicked(button):
    go_to_case(state["case_index"] - 1)


def on_next_clicked(button):
    go_to_case(state["case_index"] + 1)


def on_save_clicked(button):
    save_current_case()


def on_selector_change(change):

    if state["loading"]:
        return

    new_case_id = change["new"]

    if new_case_id not in case_ids:
        return

    index = case_ids.index(new_case_id)

    if index != state["case_index"]:
        go_to_case(index)


def on_prediction_toggle(change):
    update_prediction_visibility()


def on_pending_toggle(change):
    update_category_visibility()


def on_mark_remaining_zero(button):

    # Ne pas remplacer les annotations déjà remplies.
    for controls in category_controls.values():

        if controls["selection"].value == "unknown":
            controls["selection"].value = "0"

    state["dirty"] = True

    status_message.value = (
        "<b>Catégories restantes mises à Non.</b> "
        "Vérifier avant d'enregistrer."
    )


previous_button.on_click(on_previous_clicked)
next_button.on_click(on_next_clicked)
save_button.on_click(on_save_clicked)

mark_all_zero_button.on_click(
    on_mark_remaining_zero
)

case_selector.observe(
    on_selector_change,
    names="value"
)

show_predictions.observe(
    on_prediction_toggle,
    names="value"
)

show_only_pending.observe(
    on_pending_toggle,
    names="value"
)

# ------------------------------------------------------------
# 15. Construire l'interface
# ------------------------------------------------------------

navigation = widgets.HBox([
    previous_button,
    next_button,
    save_button
])

options = widgets.VBox([
    show_predictions,
    show_only_pending,
    mark_all_zero_button
])

interface = widgets.VBox([
    title,
    progress,
    case_selector,
    description,
    options,
    category_container,
    navigation,
    status_message
])

# ------------------------------------------------------------
# 16. Affichage initial
# ------------------------------------------------------------

load_case(0)

display(interface)

print(
    "Interface prête. "
    "Les modifications sont enregistrées "
    "avec le bouton Enregistrer ou "
    "lors d'un changement de dossier."
)

print("Fichier :", REVIEW_FILE)
print("Sauvegarde initiale :", BACKUP_FILE)


### 16. Évaluation des annotations NLP — ensemble de développement

Comparaison des prédictions automatiques aux 600 annotations humaines. Les scores sont calculés sur la classe positive et les erreurs sont exportées pour analyse. L’ensemble de test reste isolé.


In [ ]:

from pathlib import Path
import pandas as pd
import shutil
from datetime import datetime

FILENAME = "nlp_manual_review_development.csv"
destination = Path("/kaggle/working") / FILENAME

required = [
    "Annotation ID",
    "Name",
    "Cluster",
    "Category",
    "Notes",
    "Predicted",
    "Human Label",
    "Review Status"
]

def verifier(path):
    df = pd.read_csv(path, encoding="utf-8-sig")

    missing_columns = [
        col for col in required
        if col not in df.columns
    ]

    if missing_columns:
        return False, df, (
            f"Colonnes absentes : {missing_columns}"
        )

    invalid = df[required].isna().any(axis=1)

    complete = (
        len(df) == 600
        and df["Annotation ID"].nunique() == 40
        and df.groupby("Annotation ID")["Category"]
              .nunique().eq(15).all()
        and df["Human Label"].isin([0, 1]).all()
        and df["Predicted"].isin([0, 1]).all()
        and df["Review Status"].eq("Reviewed").all()
        and not invalid.any()
        and not df.duplicated(
            ["Annotation ID", "Category"]
        ).any()
    )

    diagnostic = (
        f"{len(df)} lignes, "
        f"{df['Annotation ID'].nunique()} dossiers, "
        f"{int(df['Human Label'].isna().sum())} "
        f"étiquettes humaines manquantes, "
        f"{int(invalid.sum())} lignes incomplètes"
    )

    return complete, df, diagnostic


sources = []

if destination.exists():
    sources.append(destination)

input_dir = Path("/kaggle/input")

if input_dir.exists():
    sources.extend(
        sorted(input_dir.rglob(FILENAME))
    )

print("=== DIAGNOSTIC DES FICHIERS ===")

valides = []

for path in sources:
    try:
        ok, df, diagnostic = verifier(path)
        print(f"\nFichier : {path}")
        print("Diagnostic :", diagnostic)
        print("Valide :", ok)

        if ok:
            valides.append(path)

    except Exception as exc:
        print(f"\nErreur de lecture : {path}")
        print(str(exc))


if destination in valides:
    print("\nSUCCÈS : le fichier existant est complet.")
    print("Aucune restauration nécessaire.")

elif valides:
    source = next(
        p for p in valides
        if p != destination
    )

    if destination.exists():
        stamp = datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )

        backup = destination.with_name(
            f"nlp_manual_review_backup_{stamp}.csv"
        )

        shutil.copy2(destination, backup)
        print("\nSauvegarde créée :", backup)

    shutil.copy2(source, destination)

    ok, df, diagnostic = verifier(destination)

    if not ok:
        raise RuntimeError(
            "La restauration n'a pas été validée."
        )

    print("\nSUCCÈS : annotations restaurées.")
    print("Source :", source)
    print("Destination :", destination)

else:
    print(
        "\nAucune copie complète trouvée."
        "\nLe fichier existant n'a pas été modifié."
        "\nIl faut importer le CSV original "
        "dans Add Input."
    )


In [ ]:
# ============================================================
# SCI1402 - ÉTAPE 17 : Évaluation NLP sur le développement
# 40 dossiers x 15 catégories ; aucune modification des règles
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FILENAME = 'nlp_manual_review_development.csv'

candidates = [OUTPUT_DIR / FILENAME]
input_dir = Path('/kaggle/input')
if input_dir.exists():
    candidates.extend(sorted(input_dir.rglob(FILENAME)))
candidates.append(Path('/mnt/data') / FILENAME)
source = next((p for p in candidates if p.is_file()), None)
if source is None:
    raise FileNotFoundError(f'{FILENAME} introuvable. Ajoute-le aux données Kaggle ou place-le dans /kaggle/working.')

review = pd.read_csv(source)
required = ['Annotation ID', 'Name', 'Cluster', 'Category', 'Notes', 'Predicted', 'Human Label', 'Review Status']
missing = [c for c in required if c not in review.columns]
if missing:
    raise ValueError(f'Colonnes manquantes : {missing}')
if review[required].isna().any().any():
    raise ValueError('Valeurs manquantes dans les colonnes obligatoires.')
if not review['Review Status'].eq('Reviewed').all():
    raise ValueError('Certaines lignes ne sont pas encore Reviewed.')
if not review['Predicted'].isin([0, 1]).all() or not review['Human Label'].isin([0, 1]).all():
    raise ValueError('Predicted et Human Label doivent contenir seulement 0 ou 1.')
if review.duplicated(['Annotation ID', 'Category']).any():
    raise ValueError('Doublons Annotation ID / Category : vérifie le fichier.')
counts = review.groupby('Annotation ID')['Category'].nunique()
if len(counts) != 40 or not counts.eq(15).all() or len(review) != 600:
    raise ValueError('Échantillon attendu : 40 dossiers, 15 catégories chacun, 600 lignes.')

review['Predicted'] = review['Predicted'].astype(int)
review['Human Label'] = review['Human Label'].astype(int)
y_true = review['Human Label']
y_pred = review['Predicted']
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
print(f'Fichier : {source}')
print(f'Dossiers : {len(counts)} | Annotations : {len(review)} | Erreurs : {fp + fn}')
print(f'TP={tp}, FP={fp}, FN={fn}, TN={tn}')
print(f'Précision positive : {p:.2%} | Rappel positif : {r:.2%} | F1 positif : {f1:.2%}')
print(f'Exactitude globale : {accuracy_score(y_true, y_pred):.2%}')

rows = []
for category, group in review.groupby('Category', sort=False):
    t, fpos, fneg, tpos = confusion_matrix(group['Human Label'], group['Predicted'], labels=[0, 1]).ravel()
    pp, rr, ff, _ = precision_recall_fscore_support(group['Human Label'], group['Predicted'], average='binary', zero_division=0)
    support = int(group['Human Label'].sum())
    rows.append({'Category': category, 'Support': support, 'TP': int(tpos), 'FP': int(fpos), 'FN': int(fneg), 'TN': int(t),
                 'Precision': pp if support or fpos else np.nan,
                 'Recall': rr if support else np.nan,
                 'F1': ff if support else np.nan})
metrics = pd.DataFrame(rows)
print('\nMÉTRIQUES PAR CATÉGORIE')
print(metrics.to_string(index=False, na_rep='N/A', float_format=lambda v: f'{v:.3f}'))

errors = review.loc[review['Predicted'] != review['Human Label']].copy()
errors['Error Type'] = np.where(errors['Predicted'].eq(1), 'False Positive', 'False Negative')
columns = ['Annotation ID', 'Name', 'Category', 'Predicted', 'Human Label', 'Error Type', 'Notes']
for extra in ['Detected Evidence', 'Context Flags', 'Human Evidence', 'Review Notes']:
    if extra in errors.columns:
        columns.append(extra)
errors = errors[columns]
print('\nERREURS À EXAMINER')
print(errors[['Name', 'Category', 'Error Type']].to_string(index=False))

metrics_path = OUTPUT_DIR / 'nlp_development_metrics_by_category.csv'
errors_path = OUTPUT_DIR / 'nlp_development_errors.csv'
metrics.to_csv(metrics_path, index=False)
errors.to_csv(errors_path, index=False)

fig, ax = plt.subplots(figsize=(5, 4))
cm = np.array([[tn, fp], [fn, tp]])
ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1], ['Prédit 0', 'Prédit 1'])
ax.set_yticks([0, 1], ['Réel 0', 'Réel 1'])
ax.set_title('NLP — Matrice de confusion (développement)')
for (i, j), value in np.ndenumerate(cm):
    ax.text(j, i, str(value), ha='center', va='center', color='white' if value > cm.max() / 2 else 'black')
fig.tight_layout()
figure_path = OUTPUT_DIR / 'nlp_development_confusion_matrix.png'
fig.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.show()
print('\nFichiers créés :')
for path in [metrics_path, errors_path, figure_path]:
    print(path)
print('\nATTENTION : ne pas ajuster les règles sur les 40 dossiers de test.')


### 17. Itération 2 — correction des règles NLP (développement seulement)
Cette section compare les règles initiales et révisées sur les 40 dossiers de développement. Le jeu de test reste isolé. Les scores de développement ne sont pas une estimation de généralisation.


In [ ]:
# SCI1402 — NLP itération 2 : développement uniquement
# Exécuter après avoir créé nlp_manual_review_development.csv.
# Ne lit pas les 40 dossiers de test.
from pathlib import Path
import re
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

ROOT = Path('/kaggle/working')
SOURCE = ROOT / 'nlp_manual_review_development.csv'
if not SOURCE.exists():
    raise FileNotFoundError(f'Fichier introuvable : {SOURCE}')
df = pd.read_csv(SOURCE, encoding='utf-8-sig', dtype={'Annotation ID': str})
CATEGORIES = [
    'Method Firearm', 'Method Knife', 'Method Strangulation',
    'Method Poisoning', 'Method Blunt Force', 'Method Other',
    'Victims Women', 'Victims Men', 'Victims Children',
    'Victims Elderly', 'Victims Sex Workers',
    'Sexual Violence Mentioned', 'Robbery Mentioned',
    'Dismemberment Mentioned', 'Accomplices Mentioned'
]
assert len(df) == 600 and df['Annotation ID'].nunique() == 40
assert not df[['Annotation ID', 'Category']].duplicated().any()
assert set(df['Category']) == set(CATEGORIES)
assert df['Human Label'].notna().all()
assert df['Human Label'].isin([0, 1]).all()
assert df['Review Status'].eq('Reviewed').all()

# Copie figée de la référence; aucune modification des étiquettes humaines.
baseline = df.copy()
baseline['Predicted'] = baseline['Predicted'].fillna(0).astype(int)

# Règles générales, fondées sur des mentions dans le texte (pas sur les noms).
RULES = {
 'Method Firearm': [r'\b(?:shot|shooting|shootings|gun|guns|gunshots?|firearms?|rifles?|revolvers?|pistols?)\b'],
 'Method Knife': [r'\b(?:stab(?:bed|bing|s)?|knives|knife|slash(?:ed|ing)?|axes?|hatchets?)\b'],
 'Method Strangulation': [r'\b(?:strangl(?:e|ed|ing|ation)|chok(?:e|ed|ing)|suffocat(?:e|ed|ion))\b'],
 'Method Poisoning': [r'\b(?:poison(?:ed|ing|er)?|arsenic|cyanide|overdos(?:e|es)|epinephrine)\b'],
 'Method Blunt Force': [r'\b(?:bludgeon(?:ed|ing)?|beaten|beat(?:ing)?\s+to\s+death|blunt\s+force|hammer|clubbed|rocks?|iron\s+bar)\b'],
 'Method Other': [r'\b(?:drown(?:ed|ing)|burn(?:ed|t)\s+alive|smother(?:ed|ing)|starv(?:e|ed|ation|ing)|fasting\s+until\s+they\s+starved)\b'],
 'Victims Women': [r'\b(?:women|woman|girls?|waitresses|female\s+(?:victims?|sex\s+workers?|patients?))\b'],
 'Victims Men': [r'\b(?:men|man|boys?|male\s+(?:victims?|patients?)|husbands?|sons?|grandfathers?|grandparents?)\b'],
 'Victims Children': [r'\b(?:children|child|bab(?:y|ies)|infants?|boys?|girls?)\b'],
 'Victims Elderly': [r'\b(?:elderly|seniors|old\s+age\s+pensioners)\b'],
 'Victims Sex Workers': [r'\b(?:sex\s+workers?|prostitutes?|sex\s+trade\s+workers?)\b'],
 'Sexual Violence Mentioned': [r'\b(?:rap(?:e|ed|ing|ist)|sexual\s+assaults?|sexual\s+violence|sexual\s+abuse|sodomy|extort\s+money\s+and\s+sexual\s+favors)\b'],
 'Robbery Mentioned': [r'\b(?:robbery|robberies|robbed|burglar(?:y|ies)|burglar|stole|stealing)\b'],
 'Dismemberment Mentioned': [r'\b(?:dismember(?:ed|ment)?|mutilat(?:ed|ion))\b'],
 'Accomplices Mentioned': [r'\b(?:accomplices?|partner\s+in\s+crime|with\s+the\s+help\s+of|co-conspirators?)\b',
                           r'\b(?:\d+|four|fifteen)\s+(?:people|persons|men|members|african-americans)\b.{0,100}\b(?:participat(?:ed|ion)|killed|murders?)\b']
}
REGEX = {k: [re.compile(p, re.I) for p in patterns] for k, patterns in RULES.items()}

# Les surnoms sont ignorés : « Ripper » et « Strangler » ne sont pas des preuves.
# Le texte hors guillemets reste analysé, même lorsque le surnom est cité.
def strip_aliases(s):
    return re.sub(r'\b(?:known as|referred to as|nicknamed)\s+["“][^"”]+["”]', '', s, flags=re.I)

def detect(text, category):
    text = strip_aliases(str(text) if pd.notna(text) else '')
    matches = [m for pattern in REGEX[category] for m in pattern.finditer(text)]
    if not matches:
        return 0

    # L'arme ayant tué l'auteur n'est pas une méthode utilisée sur ses victimes.
    if category == 'Method Firearm':
        matches = [m for m in matches if not re.search(
            r'\b(?:he|she|the\s+killer|the\s+murderer)\s+was\s+shot\s+dead\b',
            text[max(0, m.start()-15):min(len(text), m.end()+75)], re.I)]
        if not matches:
            return 0

    # Un homme identifié comme auteur n'est pas automatiquement une victime.
    if category == 'Victims Men':
        matches = [m for m in matches if not (
            m.group().lower() == 'man' and
            re.search(r'\b(?:romanian|homeless|american|african)\s+$',
                      text[max(0, m.start()-30):m.start()], re.I)
        )]
        if not matches:
            return 0

    # « at age 15 ... juvenile » décrit l'âge de Kemper, pas celui des victimes.
    if category == 'Victims Children':
        matches = [m for m in matches if not (
            m.group().lower() == 'juvenile' and
            re.search(r'\b(?:he|she)\s+(?:was|served)\b', text[max(0,m.start()-80):m.start()], re.I)
        )]
        if not matches:
            return 0

    # Le travail du sexe d'une dirigeante n'est pas celui des victimes.
    if category == 'Victims Sex Workers':
        matches = [m for m in matches if not re.search(
            r'\b(?:took\s+over\s+as\s+high\s+priestess|became\s+high\s+priestess)',
            text[m.end():m.end()+95], re.I)]
        if not matches:
            return 0

    return 1

# Les lignes de développement sont les seules utilisées pour l'évaluation.
iteration = baseline.copy()
iteration['Predicted V2'] = [detect(r.Notes, r.Category) for r in iteration.itertuples(index=False)]

def metrics(prediction_column):
    rows = []
    for category in CATEGORIES:
        part = iteration[iteration['Category'] == category]
        y = part['Human Label'].astype(int)
        p = part[prediction_column].astype(int)
        tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
        precision, recall, f1, _ = precision_recall_fscore_support(
            y, p, average='binary', zero_division=0)
        rows.append(dict(Category=category, Support=int(y.sum()), TP=int(tp),
                         FP=int(fp), FN=int(fn), TN=int(tn),
                         Precision=precision, Recall=recall, F1=f1))
    return pd.DataFrame(rows)

before = metrics('Predicted')
after = metrics('Predicted V2')
comparison = before[['Category','F1','FN','FP']].merge(
    after[['Category','F1','FN','FP']], on='Category', suffixes=(' V1',' V2'))
comparison['Delta F1'] = comparison['F1 V2'] - comparison['F1 V1']

for name, col in [('V1', 'Predicted'), ('V2', 'Predicted V2')]:
    y = iteration['Human Label'].astype(int)
    p = iteration[col].astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y, p, average='binary', zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0,1]).ravel()
    print(f'{name}: TP={tp}, FP={fp}, FN={fn}, TN={tn} | '
          f'précision={precision:.2%}, rappel={recall:.2%}, F1={f1:.2%}')

print('\nCOMPARAISON PAR CATÉGORIE')
print(comparison.to_string(index=False, float_format=lambda v: f'{v:.3f}'))
errors = iteration[iteration['Predicted V2'] != iteration['Human Label']].copy()
errors['Error Type'] = errors.apply(
    lambda r: 'False Positive' if r['Predicted V2'] == 1 else 'False Negative', axis=1)
print('\nERREURS RESTANTES :', len(errors))
print(errors[['Name','Category','Error Type']].to_string(index=False))
comparison.to_csv(ROOT / 'nlp_development_v1_vs_v2.csv', index=False, encoding='utf-8-sig')
errors.to_csv(ROOT / 'nlp_development_errors_v2.csv', index=False, encoding='utf-8-sig')
iteration.to_csv(ROOT / 'nlp_development_predictions_v2.csv', index=False, encoding='utf-8-sig')
print('\nExports créés dans', ROOT)


### 18. Gel des règles et prédictions sur les 40 dossiers de test

Cette étape reprend **sans changement** les règles V2 et produit 600 prédictions figées, avant toute annotation humaine du test. Le fichier est protégé contre l’écrasement et un manifeste SHA-256 permet d’en conserver la trace. **Ne pas ajuster les règles sur les résultats du test.**


In [ ]:
# SCI1402 — Étape 19 : prédictions figées sur le jeu de test
# Exécuter APRÈS les étapes précédentes du notebook.
# Aucun accès aux annotations humaines de test.
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re
import pandas as pd

ROOT = Path('/kaggle/working')
TEST_FILE = ROOT / 'annotation_test.csv'
PRED_FILE = ROOT / 'nlp_test_predictions_v2_frozen.csv'
MANIFEST_FILE = ROOT / 'nlp_test_v2_manifest.json'

CATEGORIES = [
    'Method Firearm', 'Method Knife', 'Method Strangulation',
    'Method Poisoning', 'Method Blunt Force', 'Method Other',
    'Victims Women', 'Victims Men', 'Victims Children',
    'Victims Elderly', 'Victims Sex Workers',
    'Sexual Violence Mentioned', 'Robbery Mentioned',
    'Dismemberment Mentioned', 'Accomplices Mentioned'
]

if not TEST_FILE.exists():
    raise FileNotFoundError(f'Fichier manquant : {TEST_FILE}. Exécute l’étape 14 du notebook.')
if PRED_FILE.exists() or MANIFEST_FILE.exists():
    raise FileExistsError('Une version figée existe déjà. Ne pas écraser les prédictions de test.')

test = pd.read_csv(TEST_FILE, encoding='utf-8-sig', dtype={'Annotation ID': str})
required = {'Annotation ID', 'Name', 'Cluster', 'Notes'}
missing = required - set(test.columns)
if missing:
    raise ValueError(f'Colonnes manquantes : {sorted(missing)}')
if len(test) != 40 or test['Annotation ID'].nunique() != 40:
    raise ValueError('Le jeu de test doit contenir 40 identifiants uniques.')
if test['Notes'].isna().any():
    raise ValueError('Notes manquantes dans le jeu de test.')
if test['Cluster'].value_counts().to_dict() != {0: 10, 1: 10, 2: 10, 3: 10}:
    raise ValueError('Répartition attendue : 10 dossiers par cluster.')
if any('Human Label' in c or 'Review Status' in c for c in test.columns):
    raise ValueError('Le fichier de test ne doit pas contenir les annotations humaines.')

# Règles générales, fondées sur des mentions dans le texte (pas sur les noms).
RULES = {
 'Method Firearm': [r'\b(?:shot|shooting|shootings|gun|guns|gunshots?|firearms?|rifles?|revolvers?|pistols?)\b'],
 'Method Knife': [r'\b(?:stab(?:bed|bing|s)?|knives|knife|slash(?:ed|ing)?|axes?|hatchets?)\b'],
 'Method Strangulation': [r'\b(?:strangl(?:e|ed|ing|ation)|chok(?:e|ed|ing)|suffocat(?:e|ed|ion))\b'],
 'Method Poisoning': [r'\b(?:poison(?:ed|ing|er)?|arsenic|cyanide|overdos(?:e|es)|epinephrine)\b'],
 'Method Blunt Force': [r'\b(?:bludgeon(?:ed|ing)?|beaten|beat(?:ing)?\s+to\s+death|blunt\s+force|hammer|clubbed|rocks?|iron\s+bar)\b'],
 'Method Other': [r'\b(?:drown(?:ed|ing)|burn(?:ed|t)\s+alive|smother(?:ed|ing)|starv(?:e|ed|ation|ing)|fasting\s+until\s+they\s+starved)\b'],
 'Victims Women': [r'\b(?:women|woman|girls?|waitresses|female\s+(?:victims?|sex\s+workers?|patients?))\b'],
 'Victims Men': [r'\b(?:men|man|boys?|male\s+(?:victims?|patients?)|husbands?|sons?|grandfathers?|grandparents?)\b'],
 'Victims Children': [r'\b(?:children|child|bab(?:y|ies)|infants?|boys?|girls?)\b'],
 'Victims Elderly': [r'\b(?:elderly|seniors|old\s+age\s+pensioners)\b'],
 'Victims Sex Workers': [r'\b(?:sex\s+workers?|prostitutes?|sex\s+trade\s+workers?)\b'],
 'Sexual Violence Mentioned': [r'\b(?:rap(?:e|ed|ing|ist)|sexual\s+assaults?|sexual\s+violence|sexual\s+abuse|sodomy|extort\s+money\s+and\s+sexual\s+favors)\b'],
 'Robbery Mentioned': [r'\b(?:robbery|robberies|robbed|burglar(?:y|ies)|burglar|stole|stealing)\b'],
 'Dismemberment Mentioned': [r'\b(?:dismember(?:ed|ment)?|mutilat(?:ed|ion))\b'],
 'Accomplices Mentioned': [r'\b(?:accomplices?|partner\s+in\s+crime|with\s+the\s+help\s+of|co-conspirators?)\b',
                           r'\b(?:\d+|four|fifteen)\s+(?:people|persons|men|members|african-americans)\b.{0,100}\b(?:participat(?:ed|ion)|killed|murders?)\b']
}
REGEX = {k: [re.compile(p, re.I) for p in patterns] for k, patterns in RULES.items()}

# Les surnoms sont ignorés : « Ripper » et « Strangler » ne sont pas des preuves.
# Le texte hors guillemets reste analysé, même lorsque le surnom est cité.
def strip_aliases(s):
    return re.sub(r'\b(?:known as|referred to as|nicknamed)\s+["“][^"”]+["”]', '', s, flags=re.I)

def detect(text, category):
    text = strip_aliases(str(text) if pd.notna(text) else '')
    matches = [m for pattern in REGEX[category] for m in pattern.finditer(text)]
    if not matches:
        return 0

    # L'arme ayant tué l'auteur n'est pas une méthode utilisée sur ses victimes.
    if category == 'Method Firearm':
        matches = [m for m in matches if not re.search(
            r'\b(?:he|she|the\s+killer|the\s+murderer)\s+was\s+shot\s+dead\b',
            text[max(0, m.start()-15):min(len(text), m.end()+75)], re.I)]
        if not matches:
            return 0

    # Un homme identifié comme auteur n'est pas automatiquement une victime.
    if category == 'Victims Men':
        matches = [m for m in matches if not (
            m.group().lower() == 'man' and
            re.search(r'\b(?:romanian|homeless|american|african)\s+$',
                      text[max(0, m.start()-30):m.start()], re.I)
        )]
        if not matches:
            return 0

    # « at age 15 ... juvenile » décrit l'âge de Kemper, pas celui des victimes.
    if category == 'Victims Children':
        matches = [m for m in matches if not (
            m.group().lower() == 'juvenile' and
            re.search(r'\b(?:he|she)\s+(?:was|served)\b', text[max(0,m.start()-80):m.start()], re.I)
        )]
        if not matches:
            return 0

    # Le travail du sexe d'une dirigeante n'est pas celui des victimes.
    if category == 'Victims Sex Workers':
        matches = [m for m in matches if not re.search(
            r'\b(?:took\s+over\s+as\s+high\s+priestess|became\s+high\s+priestess)',
            text[m.end():m.end()+95], re.I)]
        if not matches:
            return 0

    return 1


# Prédictions indépendantes : aucune métrique ni correction basée sur le test.
rows = []
for case in test.itertuples(index=False):
    for category in CATEGORIES:
        rows.append({
            'Annotation ID': None,
            'Name': case.Name,
            'Cluster': case.Cluster,
            'Category': category,
            'Notes': case.Notes,
            'Predicted V2': detect(case.Notes, category)
        })
# Correction de l'accès à Annotation ID (nom de colonne avec espace).
ids = test['Annotation ID'].tolist()
for i, row in enumerate(rows):
    row['Annotation ID'] = ids[i // len(CATEGORIES)]

predictions = pd.DataFrame(rows)
assert len(predictions) == 600
assert not predictions[['Annotation ID', 'Category']].duplicated().any()
assert predictions['Predicted V2'].isin([0, 1]).all()

# Empreinte des règles et des prédictions pour tracer la version évaluée.
rule_bytes = json.dumps(RULES, sort_keys=True, ensure_ascii=False).encode('utf-8')
rule_sha256 = hashlib.sha256(rule_bytes).hexdigest()
output_bytes = predictions.to_csv(index=False).encode('utf-8-sig')
output_sha256 = hashlib.sha256(output_bytes).hexdigest()

# Création exclusive : ne remplace pas un fichier déjà figé.
with PRED_FILE.open('xb') as handle:
    handle.write(output_bytes)
manifest = {
    'project': 'SCI1402',
    'version': 'NLP V2 frozen',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'test_cases': 40,
    'categories': len(CATEGORIES),
    'predictions': len(predictions),
    'rules_sha256': rule_sha256,
    'predictions_sha256': output_sha256,
    'test_source': str(TEST_FILE),
    'prediction_file': str(PRED_FILE),
    'note': 'Prédictions figées avant annotation humaine du test. Ne pas ajuster les règles sur ce jeu.'
}
with MANIFEST_FILE.open('x', encoding='utf-8') as handle:
    json.dump(manifest, handle, ensure_ascii=False, indent=2)

print(f'Dossiers de test : {len(test)} | Prédictions figées : {len(predictions)}')
print(f'Règles SHA256 : {rule_sha256}')
print(f'Prédictions SHA256 : {output_sha256}')
print(f'Fichier : {PRED_FILE}')
print(f'Manifeste : {MANIFEST_FILE}')
print('Aucune métrique calculée : annotations humaines de test non utilisées.')


In [ ]:

from pathlib import Path
import pandas as pd
import json
import hashlib

# Fichiers de validation indépendante
pred_file = Path(
    "/kaggle/working/nlp_test_predictions_v2_frozen.csv"
)

manifest_file = Path(
    "/kaggle/working/nlp_test_v2_manifest.json"
)

print("=== VÉRIFICATION DES PRÉDICTIONS FIGÉES ===")

if not pred_file.exists():
    print("ERREUR : fichier de prédictions introuvable.")

else:
    df = pd.read_csv(pred_file)

    print("Fichier :", pred_file)
    print("Nombre de lignes :", len(df))
    print("Colonnes :", df.columns.tolist())
    print("Valeurs manquantes par colonne :")
    print(df.isna().sum().to_string())

    # Empreinte du fichier existant
    sha256 = hashlib.sha256(
        pred_file.read_bytes()
    ).hexdigest()

    print("\nSHA-256 :", sha256)

    # Vérification du manifeste
    if manifest_file.exists():
        print("\nManifeste trouvé.")

        with manifest_file.open(
            "r", encoding="utf-8"
        ) as f:
            manifest = json.load(f)

        print(json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ))

    else:
        print("\nATTENTION : manifeste absent.")

    # Affichage d'un aperçu
    print("\nPremières prédictions :")
    print(df.head(10).to_string(index=False))


### Évaluation finale sur le jeu de test

La cellule suivante conserve la procédure d'évaluation originale. Elle nécessite notamment `annotation_test.csv` et `sci1402_nlp_test_frozen.py` dans `/kaggle/working`. Les règles figées ne doivent pas être modifiées en fonction des résultats du test.


In [ ]:

import pandas as pd
import hashlib
import json
import re

from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support

# ============================================================
# SCI1402 — ÉVALUATION FINALE DU NLP V2
# 40 dossiers de test × 15 catégories = 600 annotations
# ============================================================

P = Path("/kaggle/working")

test = pd.read_csv(
    P / "annotation_test.csv",
    encoding="utf-8-sig",
    dtype={"Annotation ID": str}
)

# Charger exclusivement les règles et la fonction detect()
# depuis le script V2 gelé.
src = (P / "sci1402_nlp_test_frozen.py").read_text(
    encoding="utf-8"
)

namespace = {"pd": pd, "re": re}

exec(
    src[
        src.index("# Règles générales"):
        src.index("# Prédictions indépendantes")
    ],
    namespace
)

CATEGORIES = [
    "Method Firearm",
    "Method Knife",
    "Method Strangulation",
    "Method Poisoning",
    "Method Blunt Force",
    "Method Other",
    "Victims Women",
    "Victims Men",
    "Victims Children",
    "Victims Elderly",
    "Victims Sex Workers",
    "Sexual Violence Mentioned",
    "Robbery Mentioned",
    "Dismemberment Mentioned",
    "Accomplices Mentioned",
]

detect = namespace["detect"]
rules = namespace["RULES"]

# ============================================================
# 1. ANNOTATIONS HUMAINES VALIDÉES
#
# Chaque numéro correspond au suffixe de Annotation ID.
# Les catégories absentes de la liste valent 0.
# ============================================================

positives = {
    3: [
        "Method Firearm",
        "Method Strangulation",
        "Victims Men",
        "Victims Elderly",
    ],
    5: ["Victims Women", "Victims Children"],
    7: [
        "Method Strangulation",
        "Victims Women",
        "Sexual Violence Mentioned",
    ],
    8: [
        "Victims Women",
        "Sexual Violence Mentioned",
        "Robbery Mentioned",
    ],
    10: ["Method Poisoning"],
    11: ["Victims Men", "Victims Children"],
    13: ["Victims Children"],
    14: ["Victims Children"],
    15: ["Method Poisoning", "Victims Elderly"],
    20: ["Victims Women"],
    21: [],
    22: ["Robbery Mentioned"],
    23: ["Accomplices Mentioned"],
    29: [
        "Sexual Violence Mentioned",
        "Robbery Mentioned",
    ],
    30: [
        "Method Strangulation",
        "Victims Elderly",
        "Robbery Mentioned",
    ],
    31: ["Victims Women"],
    32: ["Method Strangulation", "Victims Women"],
    34: [
        "Victims Men",
        "Victims Children",
        "Sexual Violence Mentioned",
    ],
    38: ["Accomplices Mentioned"],
    39: [
        "Victims Sex Workers",
        "Dismemberment Mentioned",
    ],
    42: ["Method Poisoning"],
    44: ["Method Firearm"],
    47: ["Method Poisoning", "Victims Children"],
    49: ["Accomplices Mentioned"],
    50: [
        "Victims Women",
        "Victims Men",
        "Accomplices Mentioned",
    ],
    52: ["Victims Sex Workers"],
    54: [],
    56: ["Method Strangulation", "Victims Women"],
    58: ["Method Poisoning"],
    60: [],
    62: ["Method Firearm"],
    63: ["Victims Women"],
    64: ["Victims Women"],
    66: ["Accomplices Mentioned"],
    68: ["Method Strangulation", "Victims Women"],
    74: ["Victims Sex Workers"],
    75: ["Victims Women"],
    76: [
        "Method Knife",
        "Method Strangulation",
        "Method Blunt Force",
        "Victims Women",
        "Victims Children",
    ],
    77: ["Victims Men"],
    80: [
        "Method Blunt Force",
        "Victims Elderly",
        "Victims Sex Workers",
        "Dismemberment Mentioned",
    ],
}

# ============================================================
# 2. NOTES MÉTHODOLOGIQUES
# ============================================================

notes_special = {
    5: "Stabbing describes death of perpetrator, not victims.",
    21: "All 15 categories absent from Notes.",
    29: "Burglary included in Robbery Mentioned.",
    32: "Sister-in-law establishes female homicide victim.",
    38: "Possible accomplice counted as mentioned.",
    39: (
        "Sex workers gender unspecified; "
        "dismemberment not necessarily killing method."
    ),
    49: "Two co-offenders explicitly identified in Name.",
    50: (
        "Two co-offenders explicitly identified in Name "
        "(user-validated exception)."
    ),
    52: "Strangler appears only in nickname.",
    54: "All 15 categories absent from Notes.",
    60: (
        "ATM targeting not explicit robbery; "
        "police shot perpetrator."
    ),
    62: (
        "Revolver explicitly used to kill; "
        "nickname not robbery evidence."
    ),
    63: (
        "Crack cocaine describes perpetrator, "
        "not poisoning method."
    ),
    64: "Strangler only in nickname.",
    66: "Female describes accomplices, not victims.",
    68: "Kidnapping/binding not an additional killing method.",
    74: (
        "Hanging describes perpetrator suicide; "
        "sex worker gender unspecified."
    ),
    75: "Two disappearances not confirmed deaths.",
    76: (
        "Unborn children included in Victims Children "
        "by explicit user validation."
    ),
    77: (
        "Hanging describes perpetrator execution; "
        "sister/nephew attacked, not confirmed killed."
    ),
    80: (
        "Decapitation counted as dismemberment; "
        "no knife instrument or distinct killing "
        "method established."
    ),
}

# ============================================================
# 3. VÉRIFICATION DES 40 DOSSIERS
# ============================================================

assert len(positives) == 40
assert len(test) == 40

ids = {
    int(s.split("-")[-1])
    for s in test["Annotation ID"]
}

assert ids == set(positives)

assert all(
    set(values) <= set(CATEGORIES)
    for values in positives.values()
)

# ============================================================
# 4. CONSTRUCTION DE LA RÉFÉRENCE HUMAINE
# ============================================================

for i, row in test.iterrows():

    num = int(row["Annotation ID"].split("-")[-1])
    validated = set(positives[num])

    for category in CATEGORIES:
        test.at[i, category] = int(category in validated)

    # Conserver le texte original comme preuve.
    test.at[i, "Evidence Quote"] = row["Notes"]

    test.at[i, "Uncertainty Notes"] = notes_special.get(
        num,
        (
            "Only explicitly described characteristics "
            "are annotated; unmentioned categories = 0."
        ),
    )

    test.at[i, "Annotation Status"] = (
        "Validated by user in conversation"
    )

assert test[CATEGORIES].notna().all().all()

reference_path = (
    P / "sci1402_test_ground_truth_40_cases.csv"
)

test.to_csv(
    reference_path,
    index=False,
    encoding="utf-8-sig",
)

# ============================================================
# 5. CONVERSION EN 600 ANNOTATIONS (FORMAT LONG)
# ============================================================

long = test.melt(
    id_vars=[
        "Annotation ID",
        "Name",
        "Cluster",
        "Notes",
        "Annotation Status",
    ],
    value_vars=CATEGORIES,
    var_name="Category",
    value_name="Human Label",
)

long["Human Label"] = long["Human Label"].astype(int)

long.to_csv(
    P / "sci1402_test_ground_truth_600_labels.csv",
    index=False,
    encoding="utf-8-sig",
)

assert len(long) == 600

# ============================================================
# 6. RECONSTITUTION DES PRÉDICTIONS V2 GELÉES
# ============================================================

rows = []

for case in test.itertuples(index=False):

    for category in CATEGORIES:

        rows.append({
            "Annotation ID": case[0],
            "Name": case.Name,
            "Cluster": case.Cluster,
            "Category": category,
            "Notes": case.Notes,
            "Predicted V2": detect(case.Notes, category),
        })

pred = pd.DataFrame(rows)

# ============================================================
# 7. CONTRÔLE D'INTÉGRITÉ SHA-256
# ============================================================

rule_hash = hashlib.sha256(
    json.dumps(
        rules,
        sort_keys=True,
        ensure_ascii=False,
    ).encode("utf-8")
).hexdigest()

pred_bytes = pred.to_csv(
    index=False
).encode("utf-8-sig")

pred_hash = hashlib.sha256(
    pred_bytes
).hexdigest()

EXPECTED_RULE_HASH = (
    "c13d52ecef8982a0da3f9a6c33f3b2250835beb208fe15ae5745f73b5a3352a6"
)

EXPECTED_PRED_HASH = (
    "22a1cbec3655c66e0196e1fc789250842d0ae0d92ac72cc75e3db117019c005f"
)

print("Rule SHA-256:", rule_hash)
print("Prediction SHA-256:", pred_hash)

if (
    rule_hash != EXPECTED_RULE_HASH
    or pred_hash != EXPECTED_PRED_HASH
):
    raise RuntimeError(
        "Les empreintes ne correspondent pas "
        "aux prédictions V2 gelées. "
        "Évaluation interrompue."
    )

print("Intégrité V2 : CONFIRMÉE")

(P / "nlp_test_predictions_v2_frozen_verified.csv").write_bytes(
    pred_bytes
)

# ============================================================
# 8. COMPARAISON PRÉDICTIONS VS ANNOTATIONS HUMAINES
# ============================================================

evaldf = pred.merge(
    long[
        [
            "Annotation ID",
            "Category",
            "Human Label",
        ]
    ],
    on=["Annotation ID", "Category"],
    validate="one_to_one",
)

def classify_outcome(row):

    human = int(row["Human Label"])
    predicted = int(row["Predicted V2"])

    if human == 1 and predicted == 1:
        return "TP"

    if human == 0 and predicted == 0:
        return "TN"

    if human == 0 and predicted == 1:
        return "FP"

    return "FN"


evaldf["Outcome"] = evaldf.apply(
    classify_outcome,
    axis=1,
)

evaldf.to_csv(
    P / "sci1402_test_v2_comparison_600.csv",
    index=False,
    encoding="utf-8-sig",
)

# ============================================================
# 9. MÉTRIQUES GLOBALES ET PAR CATÉGORIE
# ============================================================

summary = []

groups = [("GLOBAL", evaldf)]

groups += [
    (
        category,
        evaldf[evaldf["Category"] == category],
    )
    for category in CATEGORIES
]

for name, subset in groups:

    y_true = subset["Human Label"]
    y_pred = subset["Predicted V2"]

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        )
    )

    counts = subset["Outcome"].value_counts()

    summary.append({
        "Category": name,
        "N": len(subset),
        "Positives human": int(y_true.sum()),
        "Positives V2": int(y_pred.sum()),
        "TP": int(counts.get("TP", 0)),
        "FP": int(counts.get("FP", 0)),
        "FN": int(counts.get("FN", 0)),
        "TN": int(counts.get("TN", 0)),
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
    })

metrics = pd.DataFrame(summary)

metrics.to_csv(
    P / "sci1402_test_v2_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

# ============================================================
# 10. EXTRACTION DES ERREURS
# ============================================================

errors = evaldf[
    evaldf["Outcome"].isin(["FP", "FN"])
].copy()

errors.to_csv(
    P / "sci1402_test_v2_errors.csv",
    index=False,
    encoding="utf-8-sig",
)

# ============================================================
# 11. MANIFESTE DE REPRODUCTIBILITÉ
# ============================================================

manifest = {
    "dataset": "SCI1402 test",
    "test_cases": 40,
    "categories": 15,
    "human_annotations": 600,
    "frozen_rule_sha256": rule_hash,
    "frozen_prediction_sha256": pred_hash,
    "frozen_prediction_hash_matches_original": True,
    "reference_file": reference_path.name,
    "note": (
        "Human annotations reconstructed from "
        "40 explicitly validated chat dossiers; "
        "V2 rules unchanged."
    ),
}

(
    P / "sci1402_test_evaluation_manifest.json"
).write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# ============================================================
# 12. AFFICHAGE DES RÉSULTATS
# ============================================================

print("\n" + "=" * 65)
print("SCI1402 — ÉVALUATION FINALE NLP V2")
print("=" * 65)

print("\nMÉTRIQUES")
print(metrics.to_string(index=False))

print("\nERREURS")
print(
    errors[
        [
            "Annotation ID",
            "Name",
            "Category",
            "Human Label",
            "Predicted V2",
            "Outcome",
        ]
    ].to_string(index=False)
)

print("\nFICHIERS GÉNÉRÉS")

for filename in [
    "sci1402_test_ground_truth_40_cases.csv",
    "sci1402_test_ground_truth_600_labels.csv",
    "nlp_test_predictions_v2_frozen_verified.csv",
    "sci1402_test_v2_comparison_600.csv",
    "sci1402_test_v2_metrics.csv",
    "sci1402_test_v2_errors.csv",
    "sci1402_test_evaluation_manifest.json",
]:
    print(P / filename)

print("\nÉvaluation terminée.")
